<a href="https://colab.research.google.com/github/hotusiki/Cheminformatics_pipeline/blob/main/Predocking_docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oncotarget docking pipeline — full fixed Colab notebook

Исправленная версия ноутбука по шагам:

1. **Install + imports**  
2. **Mount Google Drive + project paths + helper functions**  
3. **Pocket tools setup**  
4. **Target resolution to experimental structures**  
5. **Routing: reference ligand vs pocket consensus**  
6. **Repaired pocket consensus**  
7. **GNINA + ligand preparation + job manifests**  
8. **Redocking calibration**  
9. **Production docking by batches**  
10. **Автоматическая добивка missing jobs для FCGR3A**  
11. **Финальный отчёт по всем docking results**

В notebook оставлены рабочие версии ячеек, включая self-contained recovery steps.

In [ ]:
# @title 00. Install dependencies + downloads + common imports

import os, sys, subprocess, shutil, json, re, textwrap, stat, tarfile, zipfile, math, time, csv, platform
from pathlib import Path

def _run(cmd, check=True, capture=True):
    print("\n>>", " ".join(map(str, cmd)))
    res = subprocess.run(cmd, text=True, capture_output=capture)
    if capture:
        if res.stdout.strip():
            print(res.stdout.strip())
        if res.stderr.strip():
            print(res.stderr.strip())
    if check and res.returncode != 0:
        raise RuntimeError(f"Command failed ({res.returncode}): {' '.join(map(str, cmd))}")
    return res

os.environ["DEBIAN_FRONTEND"] = "noninteractive"
_run(["apt-get", "update", "-y"])
_run([
    "apt-get", "install", "-y", "--no-install-recommends",
    "git", "wget", "curl", "unzip", "openbabel", "default-jre-headless",
    "build-essential", "openjdk-17-jre-headless", "libnetcdf-dev",
    "libstdc++6", "make", "gcc", "g++"
])

_run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
_run([
    sys.executable, "-m", "pip", "install", "-U",
    "numpy", "scipy", "pandas", "requests", "tqdm", "networkx",
    "biopython", "gemmi", "pdb2pqr", "openmm", "mdtraj", "rdkit",
    "ipywidgets", "ipykernel"
])
_run([sys.executable, "-m", "pip", "install", "-U", "git+https://github.com/openmm/pdbfixer.git"])

import io
import gzip
import itertools
from datetime import datetime
from collections import Counter, defaultdict
from urllib.parse import quote

import numpy as np
import pandas as pd
import requests

from IPython.display import display, HTML, Markdown

from Bio.PDB import PDBIO, PDBParser, MMCIFParser, Select
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign, rdMolDescriptors
from pdbfixer import PDBFixer
from openmm.app import PDBFile

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Install + imports completed.")

In [ ]:
# @title 01. Mount Google Drive + project paths + common helper functions

from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/oncotarget_pipeline')
DIRS = {
    'project_root': PROJECT_ROOT,
    'logs': PROJECT_ROOT / 'logs',
    'metadata': PROJECT_ROOT / 'metadata',
    'structures': PROJECT_ROOT / 'structures',
    'routing': PROJECT_ROOT / 'routing',
    'tools': PROJECT_ROOT / 'tools',
    'consensus_repaired': PROJECT_ROOT / 'consensus_repaired',
    'docking_setup': PROJECT_ROOT / 'docking_setup',
    'redocking_calibration': PROJECT_ROOT / 'redocking_calibration',
    'docking_runs': PROJECT_ROOT / 'docking_runs',
    'final_docking_report': PROJECT_ROOT / 'final_docking_report',
}
for p in DIRS.values():
    Path(p).mkdir(parents=True, exist_ok=True)

RUNTIME_ROOT = Path('/content/oncotarget_runtime')
RUNTIME_TOOLS_DIR = RUNTIME_ROOT / 'tools'
RUNTIME_BIN_DIR = RUNTIME_TOOLS_DIR / 'bin'
RUNTIME_SRC_DIR = RUNTIME_TOOLS_DIR / 'src'
RUNTIME_BUILD_DIR = RUNTIME_TOOLS_DIR / 'build'
for p in [RUNTIME_ROOT, RUNTIME_TOOLS_DIR, RUNTIME_BIN_DIR, RUNTIME_SRC_DIR, RUNTIME_BUILD_DIR]:
    p.mkdir(parents=True, exist_ok=True)
os.environ['PATH'] = f"{RUNTIME_BIN_DIR}:{os.environ['PATH']}"

print('Mounted at /content/drive')
print('Project root:', PROJECT_ROOT)
print('Runtime bin:', RUNTIME_BIN_DIR)

def run_cmd(cmd, check=True, cwd=None, env=None):
    print("\n>>", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, text=True, capture_output=True, cwd=cwd, env=env)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {' '.join(map(str, cmd))}")
    return result

def sanitize_name(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("_") or "unknown"

def split_csv(text):
    return [x.strip() for x in str(text).split(",") if x.strip()]

def html_escape(text):
    return str(text).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

def show_status_panel(status_list, ok_text="OK", bad_text="NOT OK"):
    rows = []
    for item in status_list:
        color = "#d1fae5" if item["ok"] else "#fee2e2"
        border = "#10b981" if item["ok"] else "#ef4444"
        icon = "✅" if item["ok"] else "❌"
        rows.append(
            f"""
            <tr>
              <td style="padding:8px 10px;border:1px solid #e5e7eb;"><b>{icon} {html_escape(item['name'])}</b></td>
              <td style="padding:8px 10px;border:1px solid #e5e7eb;background:{color};border-left:4px solid {border};">
                {html_escape(item['details'])}
              </td>
            </tr>
            """
        )
    all_ok = all(x["ok"] for x in status_list) if status_list else False
    header_color = "#ecfdf5" if all_ok else "#fef2f2"
    header_border = "#10b981" if all_ok else "#ef4444"
    header_text = f"✅ {ok_text}" if all_ok else f"❌ {bad_text}"
    html = f"""
    <div style="font-family:Arial,sans-serif;border:2px solid {header_border};border-radius:12px;overflow:hidden;margin:12px 0;">
      <div style="padding:12px 16px;background:{header_color};font-weight:700;font-size:16px;">{header_text}</div>
      <table style="width:100%;border-collapse:collapse;font-size:14px;">{''.join(rows)}</table>
    </div>
    """
    display(HTML(html))

def get_parser_for_path(path):
    path = Path(path)
    if path.suffix.lower() in {".cif", ".mmcif"}:
        return MMCIFParser(QUIET=True)
    return PDBParser(QUIET=True)

def normalize_structure_to_pdb(input_path, output_path=None):
    input_path = Path(input_path)
    output_path = Path(output_path) if output_path is not None else input_path.with_suffix(".normalized.pdb")
    parser = get_parser_for_path(input_path)
    structure = parser.get_structure("norm", str(input_path))
    io_obj = PDBIO()
    io_obj.set_structure(structure)
    io_obj.save(str(output_path))
    return str(output_path)

COMMON_EXCLUDED_HETS = {
    "HOH","WAT","DOD","EDO","GOL","PEG","MPD","DMS","ACT","ACY","FMT","EOH","IPA","MES","TRS","SO4","PO4","ACE","ACN",
    "NH4","BME","TLA","CIT","IMD","MSE","NA","K","CL","CA","MG","ZN","MN","FE","CU","CO","NI","CD","HG","SR","CS","LI","RB",
    "NAG","NDG","BMA","MAN","GAL","GLC","FUC","FUL","SIA","NAN","A2G","NGA","BGN","GCU","XYS","BGC"
}

def list_reference_ligands(structure_path):
    structure_path = Path(structure_path)
    parser = get_parser_for_path(structure_path)
    structure = parser.get_structure("ligands", str(structure_path))
    ligands = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if not str(residue.id[0]).startswith("H_"):
                    continue
                resname = residue.resname.strip()
                if resname in COMMON_EXCLUDED_HETS:
                    continue
                ligands.append({
                    "chain": chain.id,
                    "resname": resname,
                    "resseq": int(residue.id[1]),
                    "icode": str(residue.id[2]).strip()
                })
    return ligands

def inspect_structure_for_preprocessing(input_structure):
    input_structure = Path(input_structure)
    parser = get_parser_for_path(input_structure)
    structure = parser.get_structure("inspect", str(input_structure))
    protein_residues = 0
    nonprotein_residues = 0
    chains = set()
    for model in structure:
        for chain in model:
            chains.add(chain.id)
            for residue in chain:
                if residue.id[0] == " ":
                    protein_residues += 1
                else:
                    nonprotein_residues += 1
    return {
        "input_structure": str(input_structure),
        "n_chains": len(chains),
        "chains": sorted(chains),
        "protein_residues": protein_residues,
        "nonprotein_residues": nonprotein_residues,
        "reference_ligands": list_reference_ligands(input_structure)
    }

def preprocess_protein(input_structure, out_dir=None, ph=7.4, keep_waters=False, build_missing_residues=True, max_missing_residues_per_gap=20):
    input_structure = Path(input_structure)
    out_dir = Path(out_dir) if out_dir is not None else input_structure.parent / f"{input_structure.stem}_preprocessed"
    out_dir.mkdir(parents=True, exist_ok=True)
    normalized_pdb = out_dir / f"{input_structure.stem}.normalized.pdb"
    clean_receptor_pdb = out_dir / f"{input_structure.stem}_clean_pH{ph}.pdb"
    normalize_structure_to_pdb(input_structure, normalized_pdb)
    fixer = PDBFixer(filename=str(normalized_pdb))
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(ph)
    with open(clean_receptor_pdb, "w") as handle:
        PDBFile.writeFile(fixer.topology, fixer.positions, handle, keepIds=True)
    if not keep_waters:
        kept = []
        with open(clean_receptor_pdb, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                if line.startswith(("ATOM", "HETATM")) and line[17:20].strip() in {"HOH", "WAT", "DOD"}:
                    continue
                kept.append(line)
        clean_receptor_pdb.write_text("".join(kept), encoding="utf-8")
    return {
        "input_structure": str(input_structure),
        "normalized_pdb": str(normalized_pdb),
        "clean_receptor_pdb": str(clean_receptor_pdb),
        "ph": ph,
        "keep_waters": keep_waters,
        "build_missing_residues": build_missing_residues,
        "max_missing_residues_per_gap": max_missing_residues_per_gap
    }

print("Available helper functions:")
print("- normalize_structure_to_pdb(input_path, output_path=None)")
print("- list_reference_ligands(structure_path)")
print("- inspect_structure_for_preprocessing(input_structure)")
print("- preprocess_protein(input_structure, out_dir=None, ph=7.4, ...)")

Mounted at /content/drive
Mounted at /content/drive
Project root: /content/drive/MyDrive/oncotarget_pipeline
Runtime bin: /content/oncotarget_runtime/tools/bin
Available helper functions:
- normalize_structure_to_pdb(input_path, output_path=None)
- list_reference_ligands(structure_path)
- inspect_structure_for_preprocessing(input_structure)
- preprocess_protein(input_structure, out_dir=None, ph=7.4, ...)


## Pocket-tool runtime setup

This cell installs and configures:

- **P2Rank**
- **DoGSite / ProteinsPlus API helpers**

`fpocket` is intentionally not used in the repaired route because it was unstable in our Colab runtime.


In [ ]:

# @title 02. Pocket-tool setup for Colab + Google Drive

TARGETS_CSV = "LAIR1, ITGAX, ITGAM, FCGR3A, PTPRC, A2AR, TGFBR1, TRPA1, RON, IL4Rα"  # @param {type:"string"}
LIGANDS_CSV = "маравирок, пексидартиниб, Q702, TM5614, нифуроксазид, пазопаниб, ивермектин"  # @param {type:"string"}
P2RANK_RELEASE = "2.5.1"  # @param ["latest", "2.5.1"]
CLEAN_RUNTIME_BEFORE_INSTALL = True  # @param {type:"boolean"}

import os, re, json, stat, tarfile, zipfile, shutil, requests
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML

assert "PROJECT_ROOT" in globals() and "run_cmd" in globals(), "Run cell 01 first."

TOOLS_DIR = PROJECT_ROOT / "tools"
TOOLS_ARCHIVE_DIR = TOOLS_DIR / "archives"
TOOLS_MANIFEST_DIR = TOOLS_DIR / "manifests"
TOOLS_LOGS_DIR = TOOLS_DIR / "logs"
METADATA_DIR = PROJECT_ROOT / "metadata"
for p in [TOOLS_DIR, TOOLS_ARCHIVE_DIR, TOOLS_MANIFEST_DIR, TOOLS_LOGS_DIR, METADATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RUNTIME_ROOT = Path("/content/oncotarget_runtime")
RUNTIME_TOOLS_DIR = RUNTIME_ROOT / "tools"
RUNTIME_BIN_DIR = RUNTIME_TOOLS_DIR / "bin"
if CLEAN_RUNTIME_BEFORE_INSTALL and RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)
for p in [RUNTIME_ROOT, RUNTIME_TOOLS_DIR, RUNTIME_BIN_DIR]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["PATH"] = f"{RUNTIME_BIN_DIR}:{os.environ['PATH']}"

STEP2_STATUS = []
def add_status(name, ok, details=""):
    STEP2_STATUS.append({"name": str(name), "ok": bool(ok), "details": str(details)})
def _esc(x): return str(x).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
def show_status_panel():
    rows = []
    for item in STEP2_STATUS:
        color = "#d1fae5" if item["ok"] else "#fee2e2"
        border = "#10b981" if item["ok"] else "#ef4444"
        icon = "✅" if item["ok"] else "❌"
        rows.append(f"<tr><td style='padding:8px 10px;border:1px solid #e5e7eb;'><b>{icon} {_esc(item['name'])}</b></td><td style='padding:8px 10px;border:1px solid #e5e7eb;background:{color};border-left:4px solid {border};'>{_esc(item['details'])}</td></tr>")
    all_ok = all(x["ok"] for x in STEP2_STATUS) if STEP2_STATUS else False
    header = "✅ POCKET TOOLS SETUP OK" if all_ok else "❌ POCKET TOOLS SETUP NOT OK"
    border = "#10b981" if all_ok else "#ef4444"
    bg = "#ecfdf5" if all_ok else "#fef2f2"
    display(HTML(f"<div style='font-family:Arial,sans-serif;border:2px solid {border};border-radius:12px;overflow:hidden;margin:12px 0;'><div style='padding:12px 16px;background:{bg};font-weight:700;font-size:16px;'>{header}</div><table style='width:100%;border-collapse:collapse;font-size:14px;'>{''.join(rows)}</table></div>"))

def split_csv(text):
    return [x.strip() for x in str(text).split(",") if x.strip()]

TARGETS = split_csv(TARGETS_CSV)
LIGANDS = split_csv(LIGANDS_CSV)
metadata_path = METADATA_DIR / "targets_ligands.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump({"timestamp": datetime.now().isoformat(), "targets": TARGETS, "ligands": LIGANDS}, f, indent=2, ensure_ascii=False)

def download_file(url, output_path, timeout=180):
    output_path = Path(output_path)
    with requests.get(url, stream=True, timeout=timeout, allow_redirects=True) as r:
        r.raise_for_status()
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
    return output_path

def extract_archive(archive_path, dest_dir):
    archive_path = Path(archive_path); dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    if archive_path.name.endswith(".tar.gz") or archive_path.name.endswith(".tgz"):
        with tarfile.open(archive_path, "r:gz") as tar:
            tar.extractall(dest_dir)
    elif archive_path.suffix.lower() == ".zip":
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(dest_dir)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")
    return dest_dir

def ensure_executable(path):
    path = Path(path)
    path.chmod(path.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

def write_wrapper(wrapper_path, real_executable):
    wrapper_path = Path(wrapper_path); real_executable = Path(real_executable)
    wrapper_path.write_text(f'#!/usr/bin/env bash\nset -e\nexec "{real_executable}" "$@"\n', encoding="utf-8")
    ensure_executable(wrapper_path)
    return wrapper_path

def find_first_executable(root_dir, name):
    hits = [c for c in Path(root_dir).rglob(name) if c.is_file()]
    return sorted(hits, key=lambda x: len(str(x)))[0] if hits else None

def github_release_json(repo, release="latest"):
    url = f"https://api.github.com/repos/{repo}/releases/latest" if release == "latest" else f"https://api.github.com/repos/{repo}/releases/tags/{release}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.json()

# P2Rank
release_json = github_release_json("rdk/p2rank", release=P2RANK_RELEASE)
assets = release_json.get("assets", [])
asset = [a for a in assets if a["name"].endswith(".tar.gz")] or [a for a in assets if a["name"].endswith(".zip")] or assets
if not asset:
    raise RuntimeError("No release assets found for P2Rank.")
asset = asset[0]
archive_path = TOOLS_ARCHIVE_DIR / asset["name"]
if not archive_path.exists():
    download_file(asset["browser_download_url"], archive_path)
local_install_dir = RUNTIME_TOOLS_DIR / f"p2rank_{release_json.get('tag_name','latest')}"
extract_archive(archive_path, local_install_dir)
prank_real = find_first_executable(local_install_dir, "prank")
if prank_real is None:
    raise FileNotFoundError("P2Rank executable 'prank' not found.")
ensure_executable(prank_real)
P2RANK_BIN = str(write_wrapper(RUNTIME_BIN_DIR / "prank", prank_real))
p2rank_check = run_cmd([P2RANK_BIN, "-v"], check=False)
add_status("P2Rank installed", True, prank_real)
add_status("P2Rank version check", True, (p2rank_check.stdout + p2rank_check.stderr).strip()[:300])

# DoGSite API helpers
DOGSITE_CREATE_URL_V1 = "https://proteins.plus/api/dogsite_rest"
DOGSITE_SWAGGER_V2 = "https://proteins.plus/api/v2/"
PPLUS_UPLOAD_PDB_URL_V1 = "https://proteins.plus/api/pdb_files_rest"

def dogsite_submit_pdbcode_v1(pdb_code, analysis_detail="1", granularity="1", ligand="", chain=""):
    payload = {
        "dogsite": {
            "pdbCode": str(pdb_code),
            "analysisDetail": str(analysis_detail),
            "bindingSitePredictionGranularity": str(granularity),
            "ligand": str(ligand),
            "chain": str(chain)
        }
    }
    r = requests.post(DOGSITE_CREATE_URL_V1, json=payload, headers={"Accept":"application/json","Content-Type":"application/json"}, timeout=120)
    try:
        data = r.json()
    except Exception:
        data = {"raw_text": r.text}
    return r.status_code, data

def dogsite_get_job_v1(job_id_or_location):
    text = str(job_id_or_location).rstrip("/")
    job_id = text.split("/")[-1]
    r = requests.get(f"{DOGSITE_CREATE_URL_V1}/{job_id}", headers={"Accept":"application/json"}, timeout=120)
    try:
        data = r.json()
    except Exception:
        data = {"raw_text": r.text}
    return r.status_code, data

def proteinsplus_upload_pdb_v1(pdb_path):
    with open(pdb_path, "rb") as fh:
        files = {"pdb_file[pathvar]": fh}
        r = requests.post(PPLUS_UPLOAD_PDB_URL_V1, files=files, headers={"Accept":"application/json"}, timeout=300)
    try:
        data = r.json()
    except Exception:
        data = {"raw_text": r.text}
    return r.status_code, data

ok_swagger = requests.get(DOGSITE_SWAGGER_V2, timeout=30).status_code < 500
add_status("DoGSite / ProteinsPlus API reachable", ok_swagger, DOGSITE_SWAGGER_V2)
add_status("Targets/ligands metadata saved", True, metadata_path)

step2_manifest_path = TOOLS_MANIFEST_DIR / "pocket_tools_setup_manifest_fixed.json"
with open(step2_manifest_path, "w", encoding="utf-8") as f:
    json.dump({
        "timestamp": datetime.now().isoformat(),
        "targets": TARGETS,
        "ligands": LIGANDS,
        "runtime_root_local": str(RUNTIME_ROOT),
        "runtime_bin_dir_local": str(RUNTIME_BIN_DIR),
        "metadata_path_drive": str(metadata_path),
        "p2rank_bin": P2RANK_BIN,
        "status": STEP2_STATUS
    }, f, indent=2, ensure_ascii=False)
add_status("Pocket tools setup manifest saved", True, step2_manifest_path)

print("\nReady.")
print("Targets:", TARGETS)
print("Ligands:", LIGANDS)
print("Runtime root:", RUNTIME_ROOT)
print("Runtime bin:", RUNTIME_BIN_DIR)
print("Metadata:", metadata_path)
print("Manifest:", step2_manifest_path)
print("\nLoaded helper functions:")
print("- dogsite_submit_pdbcode_v1(...)")
print("- dogsite_get_job_v1(...)")
print("- proteinsplus_upload_pdb_v1(...)")
show_status_panel()


/tmp/ipykernel_3453/3737496023.py:74: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dest_dir)



>> /content/oncotarget_runtime/tools/bin/prank -v
P2Rank 2.5.1

Home: /content/oncotarget_runtime/tools/p2rank_2.5.1/p2rank_2.5.1
JVM: OpenJDK 64-Bit Server VM 17.0.18+8-Ubuntu-122.04.1 Ubuntu
OS: Linux (Version: 6.6.113+, Arch: amd64)
CPUs: 2
Max Memory: 2 GiB

Ready.
Targets: ['LAIR1', 'ITGAX', 'ITGAM', 'FCGR3A', 'PTPRC', 'A2AR', 'TGFBR1', 'TRPA1', 'RON', 'IL4Rα']
Ligands: ['маравирок', 'пексидартиниб', 'Q702', 'TM5614', 'нифуроксазид', 'пазопаниб', 'ивермектин']
Runtime root: /content/oncotarget_runtime
Runtime bin: /content/oncotarget_runtime/tools/bin
Metadata: /content/drive/MyDrive/oncotarget_pipeline/metadata/targets_ligands.json
Manifest: /content/drive/MyDrive/oncotarget_pipeline/tools/manifests/pocket_tools_setup_manifest_fixed.json

Loaded helper functions:
- dogsite_submit_pdbcode_v1(...)
- dogsite_get_job_v1(...)
- proteinsplus_upload_pdb_v1(...)


✅ P2Rank installed,/content/oncotarget_runtime/tools/p2rank_2.5.1/p2rank_2.5.1/prank
✅ P2Rank version check,"P2Rank 2.5.1 Home: /content/oncotarget_runtime/tools/p2rank_2.5.1/p2rank_2.5.1 JVM: OpenJDK 64-Bit Server VM 17.0.18+8-Ubuntu-122.04.1 Ubuntu OS: Linux (Version: 6.6.113+, Arch: amd64) CPUs: 2 Max Memory: 2 GiB"
✅ DoGSite / ProteinsPlus API reachable,https://proteins.plus/api/v2/
✅ Targets/ligands metadata saved,/content/drive/MyDrive/oncotarget_pipeline/metadata/targets_ligands.json
✅ Pocket tools setup manifest saved,/content/drive/MyDrive/oncotarget_pipeline/tools/manifests/pocket_tools_setup_manifest_fixed.json


## Target resolution and routing

The next two cells:
- map targets to reviewed human UniProt entries,
- fetch experimental PDB structures,
- detect reference ligands,
- preprocess receptors,
- decide whether each target goes through the **reference-ligand route** or the **pocket-consensus route**.


In [ ]:

# @title 03. Resolve targets to reviewed UniProt + experimental structures

import re, json, requests
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML

assert "PROJECT_ROOT" in globals() and "metadata_path" in globals(), "Run cells 01-02 first."

ORGANISM_ID = 9606  # @param {type:"integer"}
MAX_UNIPROT_HITS = 10  # @param {type:"integer"}
MAX_PDB_PER_TARGET = 5  # @param {type:"integer"}
DOWNLOAD_FORMAT = "cif"  # @param ["cif", "pdb"]
DOWNLOAD_ALL_CANDIDATES = True  # @param {type:"boolean"}
TARGET_ALIAS_OVERRIDES_JSON = """{"A2AR":"ADORA2A","RON":"MST1R","IL4Rα":"IL4RA"}"""  # @param {type:"string"}

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata_payload = json.load(f)
TARGETS = metadata_payload["targets"]
LIGANDS = metadata_payload["ligands"]
TARGET_ALIAS_OVERRIDES = json.loads(TARGET_ALIAS_OVERRIDES_JSON)

STRUCTURES_DIR = PROJECT_ROOT / "structures"
RAW_STRUCTURES_DIR = STRUCTURES_DIR / "raw"
STRUCTURE_REPORTS_DIR = STRUCTURES_DIR / "reports"
STRUCTURE_MANIFESTS_DIR = STRUCTURES_DIR / "manifests"
for p in [STRUCTURES_DIR, RAW_STRUCTURES_DIR, STRUCTURE_REPORTS_DIR, STRUCTURE_MANIFESTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

STEP3_STATUS = []
def add_status(name, ok, details=""):
    STEP3_STATUS.append({"name": str(name), "ok": bool(ok), "details": str(details)})
def _esc(x): return str(x).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
def show_status_panel():
    rows = []
    for item in STEP3_STATUS:
        color = "#d1fae5" if item["ok"] else "#fee2e2"
        border = "#10b981" if item["ok"] else "#ef4444"
        icon = "✅" if item["ok"] else "❌"
        rows.append(f"<tr><td style='padding:8px 10px;border:1px solid #e5e7eb;'><b>{icon} {_esc(item['name'])}</b></td><td style='padding:8px 10px;border:1px solid #e5e7eb;background:{color};border-left:4px solid {border};'>{_esc(item['details'])}</td></tr>")
    all_ok = all(x["ok"] for x in STEP3_STATUS) if STEP3_STATUS else False
    display(HTML(f"<div style='font-family:Arial,sans-serif;border:2px solid {'#10b981' if all_ok else '#ef4444'};border-radius:12px;overflow:hidden;margin:12px 0;'><div style='padding:12px 16px;background:{'#ecfdf5' if all_ok else '#fef2f2'};font-weight:700;font-size:16px;'>{'✅ TARGET/STRUCTURE RESOLUTION OK' if all_ok else '❌ TARGET/STRUCTURE RESOLUTION NOT OK'}</div><table style='width:100%;border-collapse:collapse;font-size:14px;'>{''.join(rows)}</table></div>"))

UNIPROT_SEARCH_URL = "https://rest.uniprot.org/uniprotkb/search"
RCSB_ENTRY_URL = "https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
RCSB_CIF_URL = "https://files.rcsb.org/download/{pdb_id}.cif"
RCSB_PDB_URL = "https://files.rcsb.org/download/{pdb_id}.pdb"

def safe_get(dct, *keys, default=None):
    cur = dct
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur

def normalize_target_name(target):
    return TARGET_ALIAS_OVERRIDES.get(str(target).strip(), str(target).strip())

def extract_gene_names(uniprot_result):
    names = []
    for g in uniprot_result.get("genes", []) or []:
        primary = safe_get(g, "geneName", "value")
        if primary:
            names.append(primary)
        for syn in g.get("synonyms", []) or []:
            if syn.get("value"):
                names.append(syn["value"])
    return sorted(list(dict.fromkeys(names)))

def extract_protein_name(uniprot_result):
    val = safe_get(uniprot_result, "proteinDescription", "recommendedName", "fullName", "value")
    if val:
        return val
    subs = safe_get(uniprot_result, "proteinDescription", "submissionNames", default=[])
    for x in subs:
        v = safe_get(x, "fullName", "value")
        if v:
            return v
    return None

def extract_pdb_crossrefs(uniprot_result):
    ids = []
    for ref in uniprot_result.get("uniProtKBCrossReferences", []) or []:
        if ref.get("database") == "PDB" and ref.get("id"):
            ids.append(ref["id"].upper())
    return sorted(list(dict.fromkeys(ids)))

def search_uniprot_for_target(target):
    normalized = normalize_target_name(target)
    query = f'(gene_exact:{normalized} OR gene:{normalized} OR protein_name:"{normalized}") AND reviewed:true AND organism_id:{ORGANISM_ID}'
    params = {
        "query": query,
        "format": "json",
        "size": MAX_UNIPROT_HITS,
        "fields": "accession,id,gene_names,protein_name,organism_name,length,xref_pdb"
    }
    r = requests.get(UNIPROT_SEARCH_URL, params=params, timeout=60)
    r.raise_for_status()
    results = r.json().get("results", []) or []
    ranked = []
    for res in results:
        genes = extract_gene_names(res)
        protein_name = extract_protein_name(res)
        pdb_ids = extract_pdb_crossrefs(res)
        exact = normalized in genes or any(normalized.lower() == g.lower() for g in genes)
        contains = protein_name is not None and normalized.lower() in protein_name.lower()
        score = (1 if exact else 0, 1 if contains else 0, len(pdb_ids))
        ranked.append({
            "accession": res.get("primaryAccession"),
            "uniprot_id": res.get("uniProtkbId"),
            "genes": genes,
            "protein_name": protein_name,
            "organism_name": safe_get(res, "organism", "scientificName"),
            "length": safe_get(res, "sequence", "length"),
            "pdb_ids": pdb_ids,
            "score": score
        })
    return sorted(ranked, key=lambda x: x["score"], reverse=True)

def fetch_entry_meta(pdb_id):
    r = requests.get(RCSB_ENTRY_URL.format(pdb_id=pdb_id.lower()), timeout=60)
    r.raise_for_status()
    return r.json()

def extract_resolution(entry_json):
    val = safe_get(entry_json, "rcsb_entry_info", "resolution_combined")
    nums = [x for x in val if isinstance(x, (int, float))] if isinstance(val, list) else []
    return min(nums) if nums else None

def extract_method(entry_json):
    methods = [x.get("method") for x in (entry_json.get("exptl", []) or []) if x.get("method")]
    return "; ".join(methods) if methods else None

def download_structure_file(pdb_id, out_dir, fmt="cif"):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    if fmt == "cif":
        url, out_path = RCSB_CIF_URL.format(pdb_id=pdb_id.upper()), out_dir / f"{pdb_id.upper()}.cif"
    else:
        url, out_path = RCSB_PDB_URL.format(pdb_id=pdb_id.upper()), out_dir / f"{pdb_id.upper()}.pdb"
    r = requests.get(url, timeout=120); r.raise_for_status()
    out_path.write_bytes(r.content)
    return out_path, url

def inspect_pdb_candidate(pdb_id, target_name):
    target_dir = RAW_STRUCTURES_DIR / re.sub(r"[^A-Za-z0-9._-]+", "_", target_name)
    file_path, source_url = download_structure_file(pdb_id, target_dir, fmt=DOWNLOAD_FORMAT)
    entry = fetch_entry_meta(pdb_id)
    return {
        "pdb_id": pdb_id.upper(),
        "file_path": str(file_path),
        "source_url": source_url,
        "resolution": extract_resolution(entry),
        "method": extract_method(entry),
        "reference_ligands": list_reference_ligands(file_path),
        "has_reference_ligand": len(list_reference_ligands(file_path)) > 0
    }

def rank_candidates(cands):
    def mrank(m):
        mt = (m or "").lower()
        if "x-ray" in mt: return 0
        if "electron microscopy" in mt or "cryo-em" in mt: return 1
        if "nmr" in mt: return 2
        return 10
    return sorted(cands, key=lambda c: (0 if c.get("has_reference_ligand") else 1, mrank(c.get("method")), c.get("resolution") if c.get("resolution") is not None else 999))

all_target_reports = []
for target in TARGETS:
    rep = {
        "target_input": target,
        "normalized_target": normalize_target_name(target),
        "timestamp": datetime.now().isoformat(),
        "uniprot_candidates": [],
        "selected_uniprot": None,
        "experimental_pdb_candidates": [],
        "selected_structure": None,
        "needs_model_fallback": False,
        "status": "started"
    }
    try:
        hits = search_uniprot_for_target(target)
        rep["uniprot_candidates"] = hits
        if not hits:
            rep["status"] = "no_uniprot_hit"
            rep["needs_model_fallback"] = True
            add_status(f"{target}: UniProt resolution", False, "No reviewed human UniProt hit found")
            all_target_reports.append(rep)
            continue
        best_hit = hits[0]
        rep["selected_uniprot"] = best_hit
        add_status(f"{target}: UniProt resolution", True, f"{best_hit['accession']} | PDB refs: {len(best_hit['pdb_ids'])}")
        pdb_ids = best_hit["pdb_ids"][:MAX_PDB_PER_TARGET] if DOWNLOAD_ALL_CANDIDATES else best_hit["pdb_ids"][:1]
        if not pdb_ids:
            rep["status"] = "no_experimental_pdb"
            rep["needs_model_fallback"] = True
            add_status(f"{target}: experimental structures", False, "No PDB cross-references on selected UniProt entry")
            all_target_reports.append(rep)
            continue
        cands = []
        for pdb_id in pdb_ids:
            try:
                cands.append(inspect_pdb_candidate(pdb_id, target))
            except Exception as e:
                cands.append({"pdb_id": pdb_id.upper(), "file_path": None, "download_error": f"{type(e).__name__}: {e}"})
        rep["experimental_pdb_candidates"] = cands
        valid = [c for c in cands if c.get("file_path")]
        if not valid:
            rep["status"] = "pdb_download_failed"
            rep["needs_model_fallback"] = True
            add_status(f"{target}: structure downloads", False, "All candidate downloads failed")
        else:
            selected = rank_candidates(valid)[0]
            rep["selected_structure"] = selected
            rep["status"] = "ok"
            add_status(f"{target}: selected structure", True, f"{selected['pdb_id']} | ligand={'yes' if selected['has_reference_ligand'] else 'no'} | method={selected['method']} | resolution={selected['resolution']}")
    except Exception as e:
        rep["status"] = "error"
        rep["needs_model_fallback"] = True
        rep["error"] = f"{type(e).__name__}: {e}"
        add_status(f"{target}: processing", False, rep["error"])
    all_target_reports.append(rep)

summary_rows = []
for rep in all_target_reports:
    sel_u = rep.get("selected_uniprot") or {}
    sel_s = rep.get("selected_structure") or {}
    summary_rows.append({
        "target_input": rep.get("target_input"),
        "normalized_target": rep.get("normalized_target"),
        "status": rep.get("status"),
        "uniprot_accession": sel_u.get("accession"),
        "selected_pdb_id": sel_s.get("pdb_id"),
        "selected_structure_file": sel_s.get("file_path"),
        "selected_has_reference_ligand": sel_s.get("has_reference_ligand"),
        "selected_resolution": sel_s.get("resolution"),
        "selected_method": sel_s.get("method"),
        "needs_model_fallback": rep.get("needs_model_fallback", False)
    })

manifest_path = STRUCTURE_MANIFESTS_DIR / "target_structure_resolution_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump({"timestamp": datetime.now().isoformat(), "all_target_reports": all_target_reports, "summary_rows": summary_rows, "status": STEP3_STATUS}, f, indent=2, ensure_ascii=False)

summary_tsv = STRUCTURE_REPORTS_DIR / "target_structure_summary.tsv"
with open(summary_tsv, "w", encoding="utf-8") as f:
    headers = list(summary_rows[0].keys()) if summary_rows else []
    f.write("\t".join(headers) + "\n")
    for row in summary_rows:
        f.write("\t".join([str(row.get(h, "")) for h in headers]) + "\n")

add_status("Structure resolution manifest saved", True, manifest_path)
add_status("Compact summary TSV saved", True, summary_tsv)

print("\nReady.")
print("Manifest:", manifest_path)
print("Summary TSV:", summary_tsv)
print("\nSelected structures:")
for row in summary_rows:
    print(f"- {row['target_input']}: UniProt={row['uniprot_accession']} | PDB={row['selected_pdb_id']} | ref_ligand={row['selected_has_reference_ligand']} | fallback={row['needs_model_fallback']}")
show_status_panel()


## Select one working structure, reference ligand, preprocess receptor, choose routing

In [ ]:
# @title 04. Select one working structure + one reference ligand + preprocess receptor + choose routing path

PH_FOR_PREPROCESS = 7.4  # @param {type:"number"}
KEEP_WATERS_IN_RECEPTOR = False  # @param {type:"boolean"}
BUILD_MISSING_RESIDUES = True  # @param {type:"boolean"}
MAX_MISSING_RESIDUES_PER_GAP = 20  # @param {type:"integer"}

LIGAND_PROTEIN_DISTANCE_CUTOFF = 5.0  # @param {type:"number"}
POCKET_CONTACT_DISTANCE = 5.0  # @param {type:"number"}
MIN_REFERENCE_LIGAND_HEAVY_ATOMS = 6  # @param {type:"integer"}
MIN_REFERENCE_LIGAND_CONTACT_RESIDUES = 3  # @param {type:"integer"}
GRID_PADDING_ANGSTROM = 6.0  # @param {type:"number"}
MIN_GRID_BOX_SIZE = 18.0  # @param {type:"number"}

STATUS = []
def add_status(name, ok, details=""):
    STATUS.append({"name": str(name), "ok": bool(ok), "details": str(details)})

STRUCTURE_MANIFESTS_DIR = DIRS["structures"] / "manifests"
step3_manifest_path = STRUCTURE_MANIFESTS_DIR / "target_structure_resolution_manifest.json"
with open(step3_manifest_path, "r", encoding="utf-8") as f:
    step3_manifest = json.load(f)
target_reports = step3_manifest.get("all_target_reports", [])

STEP4_DIR = DIRS["routing"]
STEP4_REPORTS_DIR = STEP4_DIR / "reports"
STEP4_MANIFESTS_DIR = STEP4_DIR / "manifests"
STEP4_RECEPTORS_DIR = STEP4_DIR / "receptors"
STEP4_LIGANDS_DIR = STEP4_DIR / "reference_ligands"
STEP4_POCKETS_DIR = STEP4_DIR / "pockets"
for p in [STEP4_REPORTS_DIR, STEP4_MANIFESTS_DIR, STEP4_RECEPTORS_DIR, STEP4_LIGANDS_DIR, STEP4_POCKETS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

COMMON_IONS = {
    "NA", "K", "CL", "CA", "MG", "ZN", "MN", "FE", "CU", "CO", "NI",
    "CD", "HG", "SR", "CS", "LI", "RB", "IOD", "BR", "F"
}
COMMON_SOLVENTS = {
    "HOH", "WAT", "DOD", "EDO", "GOL", "PEG", "MPD", "DMS", "ACT",
    "ACY", "FMT", "EOH", "IPA", "MES", "TRS", "SO4", "PO4", "ACE",
    "ACN", "NH4", "BME", "TLA", "CIT", "IMD", "MSE"
}
COMMON_GLYCANS = {
    "NAG", "NDG", "BMA", "MAN", "GAL", "GLC", "FUC", "FUL", "SIA",
    "NAN", "A2G", "NGA", "BGN", "GCU", "XYS", "BGC"
}
EXCLUDED_HETS = COMMON_IONS | COMMON_SOLVENTS | COMMON_GLYCANS

def residue_identifier(residue, chain_id):
    _, resseq, icode = residue.id
    return {"chain": chain_id, "resname": residue.resname.strip(), "resseq": int(resseq), "icode": str(icode).strip()}

def is_protein_residue(residue):
    return residue.id[0] == " "

def atom_is_hydrogen(atom):
    element = getattr(atom, "element", None)
    if element is None:
        return atom.get_name().strip().upper().startswith("H")
    return str(element).upper() == "H"

def residue_heavy_atom_coords(residue):
    coords, atom_names = [], []
    for atom in residue.get_atoms():
        if atom_is_hydrogen(atom):
            continue
        coords.append(atom.coord)
        atom_names.append(atom.get_name())
    if not coords:
        return np.empty((0, 3)), atom_names
    return np.array(coords, dtype=float), atom_names

def collect_protein_residues_and_atoms(structure):
    protein_residues = []
    protein_atom_coords = []
    protein_atom_map = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if not is_protein_residue(residue):
                    continue
                protein_residues.append((chain.id, residue))
                for atom in residue.get_atoms():
                    if atom_is_hydrogen(atom):
                        continue
                    protein_atom_coords.append(atom.coord)
                    protein_atom_map.append({
                        "chain": chain.id,
                        "resname": residue.resname.strip(),
                        "resseq": int(residue.id[1]),
                        "icode": str(residue.id[2]).strip()
                    })
    protein_atom_coords = np.array(protein_atom_coords, dtype=float) if protein_atom_coords else np.empty((0, 3))
    return protein_residues, protein_atom_coords, protein_atom_map

def analyze_ligands_in_structure(structure, pocket_cutoff=5.0):
    _, protein_atom_coords, protein_atom_map = collect_protein_residues_and_atoms(structure)
    ligands = []
    for model in structure:
        for chain in model:
            for residue in chain:
                hetflag = residue.id[0]
                if not str(hetflag).startswith("H_"):
                    continue
                resname = residue.resname.strip()
                lig_info = residue_identifier(residue, chain.id)
                coords, _ = residue_heavy_atom_coords(residue)
                heavy_atoms = int(coords.shape[0])

                if heavy_atoms == 0:
                    lig_info.update({
                        "heavy_atom_count": 0,
                        "total_atom_count": sum(1 for _ in residue.get_atoms()),
                        "min_distance_to_protein": None,
                        "contact_residue_count": 0,
                        "contact_residues": [],
                        "centroid": None,
                        "bbox_min": None,
                        "bbox_max": None,
                        "bbox_span": None,
                        "excluded_by_name": resname in EXCLUDED_HETS
                    })
                    ligands.append(lig_info)
                    continue

                min_dist = None
                contact_residues = []
                if protein_atom_coords.shape[0] > 0:
                    d = np.sqrt(((coords[:, None, :] - protein_atom_coords[None, :, :]) ** 2).sum(axis=2))
                    min_dist = float(np.min(d))
                    close_atom_idx = np.where(np.min(d, axis=0) <= pocket_cutoff)[0]
                    seen = set()
                    residue_set = []
                    for idx in close_atom_idx:
                        r = protein_atom_map[idx]
                        key = (r["chain"], r["resname"], r["resseq"], r["icode"])
                        if key in seen:
                            continue
                        seen.add(key)
                        residue_set.append(r)
                    contact_residues = residue_set

                centroid = coords.mean(axis=0)
                bbox_min = coords.min(axis=0)
                bbox_max = coords.max(axis=0)
                bbox_span = bbox_max - bbox_min
                lig_info.update({
                    "heavy_atom_count": heavy_atoms,
                    "total_atom_count": sum(1 for _ in residue.get_atoms()),
                    "min_distance_to_protein": min_dist,
                    "contact_residue_count": len(contact_residues),
                    "contact_residues": contact_residues,
                    "centroid": [float(x) for x in centroid],
                    "bbox_min": [float(x) for x in bbox_min],
                    "bbox_max": [float(x) for x in bbox_max],
                    "bbox_span": [float(x) for x in bbox_span],
                    "excluded_by_name": resname in EXCLUDED_HETS
                })
                ligands.append(lig_info)
    return ligands

def classify_reference_ligand_candidate(lig):
    reasons = []
    ok = True
    if lig["resname"] in EXCLUDED_HETS:
        ok = False
        reasons.append("excluded_common_het")
    if lig["heavy_atom_count"] < MIN_REFERENCE_LIGAND_HEAVY_ATOMS:
        ok = False
        reasons.append("too_few_heavy_atoms")
    md = lig.get("min_distance_to_protein")
    if md is None or md > LIGAND_PROTEIN_DISTANCE_CUTOFF:
        ok = False
        reasons.append("not_close_to_protein")
    if lig.get("contact_residue_count", 0) < MIN_REFERENCE_LIGAND_CONTACT_RESIDUES:
        ok = False
        reasons.append("too_few_contact_residues")
    return {"is_candidate": ok, "reasons": reasons}

def ligand_priority_score(lig):
    md = lig.get("min_distance_to_protein")
    if md is None:
        md = 999.0
    return (lig.get("contact_residue_count", 0), lig.get("heavy_atom_count", 0), -float(md))

def choose_best_reference_ligand(ligands):
    candidates = []
    for lig in ligands:
        cls = classify_reference_ligand_candidate(lig)
        lig2 = dict(lig)
        lig2["candidate_classification"] = cls
        if cls["is_candidate"]:
            candidates.append(lig2)
    if not candidates:
        return None, []
    candidates = sorted(candidates, key=ligand_priority_score, reverse=True)
    return candidates[0], candidates

class SingleResidueSelect(Select):
    def __init__(self, chain_id, residue_id):
        super().__init__()
        self.chain_id = chain_id
        self.residue_id = residue_id
    def accept_chain(self, chain):
        return 1 if chain.id == self.chain_id else 0
    def accept_residue(self, residue):
        return 1 if residue.id == self.residue_id else 0

def export_residue_to_pdb(structure, chain_id, residue_id, out_path):
    io_obj = PDBIO()
    io_obj.set_structure(structure)
    io_obj.save(str(out_path), SingleResidueSelect(chain_id, residue_id))
    return out_path

def compute_grid_from_ligand(lig):
    center = lig["centroid"]
    span = lig["bbox_span"]
    size = [max(float(s) + float(GRID_PADDING_ANGSTROM), float(MIN_GRID_BOX_SIZE)) for s in span]
    return {
        "center_x": float(center[0]),
        "center_y": float(center[1]),
        "center_z": float(center[2]),
        "size_x": float(size[0]),
        "size_y": float(size[1]),
        "size_z": float(size[2])
    }

routing_reports = []
for rep in target_reports:
    target_name = rep.get("target_input")
    safe_target = sanitize_name(target_name)
    route_report = {
        "target_input": target_name,
        "normalized_target": rep.get("normalized_target"),
        "timestamp": datetime.now().isoformat(),
        "selected_structure_from_step3": rep.get("selected_structure"),
        "selected_uniprot_from_step3": rep.get("selected_uniprot"),
        "routing_decision": None,
        "selected_reference_ligand": None,
        "all_detected_ligands": [],
        "candidate_reference_ligands": [],
        "preprocess_result": None,
        "reference_ligand_pdb": None,
        "reference_ligand_grid": None,
        "reference_ligand_contact_residues": [],
        "status": "started"
    }
    try:
        selected_structure = rep.get("selected_structure")
        needs_model_fallback = rep.get("needs_model_fallback", False)
        if needs_model_fallback or not selected_structure or not selected_structure.get("file_path"):
            route_report["routing_decision"] = "model_fallback_required"
            route_report["status"] = "model_fallback_required"
            add_status(f"{target_name}: routing", True, "model_fallback_required")
            routing_reports.append(route_report)
            continue

        structure_file = Path(selected_structure["file_path"])
        structure = get_parser_for_path(structure_file).get_structure(safe_target, str(structure_file))
        ligands = analyze_ligands_in_structure(structure, pocket_cutoff=POCKET_CONTACT_DISTANCE)
        route_report["all_detected_ligands"] = ligands
        best_ref_lig, ref_candidates = choose_best_reference_ligand(ligands)
        route_report["candidate_reference_ligands"] = ref_candidates

        receptor_out_dir = STEP4_RECEPTORS_DIR / safe_target
        receptor_out_dir.mkdir(parents=True, exist_ok=True)
        preprocess_result = preprocess_protein(
            input_structure=structure_file,
            out_dir=receptor_out_dir,
            ph=PH_FOR_PREPROCESS,
            keep_waters=KEEP_WATERS_IN_RECEPTOR,
            build_missing_residues=BUILD_MISSING_RESIDUES,
            max_missing_residues_per_gap=MAX_MISSING_RESIDUES_PER_GAP
        )
        route_report["preprocess_result"] = preprocess_result

        if best_ref_lig is not None:
            lig_chain = best_ref_lig["chain"]
            lig_resname = best_ref_lig["resname"]
            lig_resseq = int(best_ref_lig["resseq"])
            lig_icode = best_ref_lig["icode"]

            selected_residue = None
            selected_chain = None
            for model in structure:
                for chain in model:
                    if chain.id != lig_chain:
                        continue
                    for residue in chain:
                        if residue.resname.strip() != lig_resname:
                            continue
                        if int(residue.id[1]) != lig_resseq:
                            continue
                        if str(residue.id[2]).strip() != str(lig_icode).strip():
                            continue
                        if not str(residue.id[0]).startswith("H_"):
                            continue
                        selected_residue = residue
                        selected_chain = chain
                        break
                    if selected_residue is not None:
                        break
                if selected_residue is not None:
                    break
            if selected_residue is None or selected_chain is None:
                raise RuntimeError(f"Could not re-find selected ligand residue in structure: {best_ref_lig}")

            ligand_out_dir = STEP4_LIGANDS_DIR / safe_target
            ligand_out_dir.mkdir(parents=True, exist_ok=True)
            ligand_pdb_path = ligand_out_dir / f"{safe_target}_{lig_resname}_{lig_chain}_{lig_resseq}.pdb"
            export_residue_to_pdb(structure, selected_chain.id, selected_residue.id, ligand_pdb_path)

            grid = compute_grid_from_ligand(best_ref_lig)
            pocket_json_path = STEP4_POCKETS_DIR / f"{safe_target}_reference_ligand_pocket.json"
            with open(pocket_json_path, "w", encoding="utf-8") as f:
                json.dump({
                    "target": target_name,
                    "selected_structure_file": str(structure_file),
                    "selected_reference_ligand": best_ref_lig,
                    "grid": grid,
                    "contact_residues": best_ref_lig["contact_residues"]
                }, f, indent=2, ensure_ascii=False)

            route_report["selected_reference_ligand"] = best_ref_lig
            route_report["reference_ligand_pdb"] = str(ligand_pdb_path)
            route_report["reference_ligand_grid"] = grid
            route_report["reference_ligand_contact_residues"] = best_ref_lig["contact_residues"]
            route_report["routing_decision"] = "reference_ligand_first"
            route_report["status"] = "ok"
            add_status(
                f"{target_name}: routing",
                True,
                f"reference_ligand_first | {best_ref_lig['resname']} chain {best_ref_lig['chain']} resseq {best_ref_lig['resseq']}"
            )
        else:
            route_report["routing_decision"] = "pocket_consensus_required"
            route_report["status"] = "ok"
            add_status(f"{target_name}: routing", True, "pocket_consensus_required")

        target_report_path = STEP4_REPORTS_DIR / f"{safe_target}_routing_report.json"
        with open(target_report_path, "w", encoding="utf-8") as f:
            json.dump(route_report, f, indent=2, ensure_ascii=False)
        route_report["report_path"] = str(target_report_path)
    except Exception as e:
        route_report["status"] = "error"
        route_report["error"] = f"{type(e).__name__}: {e}"
        add_status(f"{target_name}: processing", False, f"{type(e).__name__}: {e}")
    routing_reports.append(route_report)

summary_rows = []
for rr in routing_reports:
    ref = rr.get("selected_reference_ligand") or {}
    prep = rr.get("preprocess_result") or {}
    grid = rr.get("reference_ligand_grid") or {}
    summary_rows.append({
        "target_input": rr.get("target_input"),
        "normalized_target": rr.get("normalized_target"),
        "routing_decision": rr.get("routing_decision"),
        "status": rr.get("status"),
        "selected_structure_file": (rr.get("selected_structure_from_step3") or {}).get("file_path"),
        "preprocessed_receptor_pdb": prep.get("clean_receptor_pdb"),
        "reference_ligand_resname": ref.get("resname"),
        "reference_ligand_chain": ref.get("chain"),
        "reference_ligand_resseq": ref.get("resseq"),
        "reference_ligand_pdb": rr.get("reference_ligand_pdb"),
        "grid_center_x": grid.get("center_x"),
        "grid_center_y": grid.get("center_y"),
        "grid_center_z": grid.get("center_z"),
        "grid_size_x": grid.get("size_x"),
        "grid_size_y": grid.get("size_y"),
        "grid_size_z": grid.get("size_z")
    })

routing_manifest_path = STEP4_MANIFESTS_DIR / "routing_manifest.json"
with open(routing_manifest_path, "w", encoding="utf-8") as f:
    json.dump({
        "timestamp": datetime.now().isoformat(),
        "routing_reports": routing_reports,
        "summary_rows": summary_rows,
        "status": STATUS
    }, f, indent=2, ensure_ascii=False)

summary_tsv_path = STEP4_REPORTS_DIR / "routing_summary.tsv"
with open(summary_tsv_path, "w", encoding="utf-8") as f:
    headers = [
        "target_input", "normalized_target", "routing_decision", "status",
        "selected_structure_file", "preprocessed_receptor_pdb",
        "reference_ligand_resname", "reference_ligand_chain", "reference_ligand_resseq",
        "reference_ligand_pdb", "grid_center_x", "grid_center_y", "grid_center_z",
        "grid_size_x", "grid_size_y", "grid_size_z"
    ]
    f.write("\t".join(headers) + "\n")
    for row in summary_rows:
        f.write("\t".join([str(row.get(h, "")) for h in headers]) + "\n")

add_status("Routing manifest saved", True, str(routing_manifest_path))
add_status("Routing summary TSV saved", True, str(summary_tsv_path))

print("\nReady.")
print("Routing manifest:", routing_manifest_path)
print("Routing summary TSV:", summary_tsv_path)
show_status_panel(STATUS, ok_text="ROUTING STEP OK", bad_text="ROUTING STEP NOT OK")

## Repaired pocket consensus

In [ ]:
# @title 05. Repaired pocket consensus: P2Rank + DoGSite + explicit P2Rank fallback

MAX_P2RANK_POCKETS = 5  # @param {type:"integer"}
MAX_DOGSITE_POCKETS = 5  # @param {type:"integer"}

DOGSITE_ANALYSIS_DETAIL = "1"  # @param ["0", "1"]
DOGSITE_GRANULARITY = "1"  # @param ["0", "1"]

DOGSITE_POLL_INTERVAL_SEC = 12  # @param {type:"integer"}
DOGSITE_MAX_POLLS = 35  # @param {type:"integer"}
DOGSITE_INITIAL_BACKOFF_SEC = 25  # @param {type:"integer"}
DOGSITE_MAX_RETRIES_429 = 4  # @param {type:"integer"}

CONSENSUS_GRID_PADDING = 6.0  # @param {type:"number"}
CONSENSUS_MIN_BOX_SIZE = 18.0  # @param {type:"number"}

ALLOW_SINGLE_METHOD_FALLBACK = True  # @param {type:"boolean"}
FORCE_RERUN_DOGSITE = False  # @param {type:"boolean"}

STATUS = []
def add_status(name, ok, details=""):
    STATUS.append({"name": str(name), "ok": bool(ok), "details": str(details)})

STEP4_MANIFESTS_DIR = DIRS["routing"] / "manifests"
routing_manifest_path = STEP4_MANIFESTS_DIR / "routing_manifest.json"
with open(routing_manifest_path, "r", encoding="utf-8") as f:
    routing_manifest = json.load(f)
routing_reports = routing_manifest.get("routing_reports", [])
targets_for_consensus = [r for r in routing_reports if r.get("routing_decision") == "pocket_consensus_required"]

STEP5R_DIR = DIRS["consensus_repaired"]
STEP5R_REPORTS_DIR = STEP5R_DIR / "reports"
STEP5R_MANIFESTS_DIR = STEP5R_DIR / "manifests"
STEP5R_METHODS_DIR = STEP5R_DIR / "methods"
STEP5R_GRIDS_DIR = STEP5R_DIR / "grids"
STEP5R_CACHE_DIR = STEP5R_DIR / "cache"
STEP5R_TMP_DIR = Path("/content/oncotarget_runtime/consensus_repaired_tmp")
for p in [STEP5R_REPORTS_DIR, STEP5R_MANIFESTS_DIR, STEP5R_METHODS_DIR, STEP5R_GRIDS_DIR, STEP5R_CACHE_DIR, STEP5R_TMP_DIR]:
    p.mkdir(parents=True, exist_ok=True)
DOGSITE_CACHE_DIR = STEP5R_CACHE_DIR / "dogsite"
DOGSITE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

add_status("Loaded routing manifest", True, str(routing_manifest_path))
add_status("Targets requiring repaired consensus", True, str(len(targets_for_consensus)))

def residue_key(chain_id, residue):
    return f"{chain_id}_{int(residue.id[1])}"

def atom_is_h(atom):
    element = getattr(atom, "element", None)
    if element is None:
        return atom.get_name().strip().upper().startswith("H")
    return str(element).upper() == "H"

def get_structure_residue_atom_map(structure):
    residue_to_coords = {}
    for model in structure:
        for chain in model:
            for residue in chain:
                if residue.id[0] != " ":
                    continue
                coords = []
                for atom in residue.get_atoms():
                    if atom_is_h(atom):
                        continue
                    coords.append(atom.coord)
                if coords:
                    residue_to_coords[residue_key(chain.id, residue)] = np.array(coords, dtype=float)
    return residue_to_coords

def compute_grid_from_residue_ids(receptor_pdb_path, residue_ids, padding=6.0, min_box=18.0):
    structure = get_parser_for_path(receptor_pdb_path).get_structure("grid", str(receptor_pdb_path))
    residue_map = get_structure_residue_atom_map(structure)
    selected = []
    for rid in residue_ids:
        arr = residue_map.get(rid)
        if arr is not None and arr.size > 0:
            selected.append(arr)
    if not selected:
        return None
    coords = np.vstack(selected)
    centroid = coords.mean(axis=0)
    bbox_min = coords.min(axis=0)
    bbox_max = coords.max(axis=0)
    span = bbox_max - bbox_min
    size = [max(float(s) + float(padding), float(min_box)) for s in span]
    return {
        "center_x": float(centroid[0]),
        "center_y": float(centroid[1]),
        "center_z": float(centroid[2]),
        "size_x": float(size[0]),
        "size_y": float(size[1]),
        "size_z": float(size[2]),
        "n_residues_used": len(residue_ids)
    }

def parse_residue_ids_from_text(text):
    if text is None:
        return set()
    text = str(text)
    patterns = [
        r"\b([A-Za-z0-9])_(-?\d+[A-Za-z]?)\b",
        r"\b([A-Za-z0-9]):(-?\d+[A-Za-z]?)\b",
        r"\bChain\s*([A-Za-z0-9])\s*[:_ ]\s*Residue\s*(-?\d+[A-Za-z]?)\b"
    ]
    hits = set()
    for pat in patterns:
        for m in re.finditer(pat, text):
            hits.add(f"{m.group(1)}_{m.group(2)}")
    return hits

def jaccard(a, b):
    a = set(a); b = set(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def find_p2rank_predictions_csv(output_dir):
    matches = list(Path(output_dir).rglob("*_predictions.csv"))
    return matches[0] if matches else None

def run_p2rank_on_receptor(receptor_pdb, target_safe_name):
    out_dir = STEP5R_METHODS_DIR / target_safe_name / "p2rank"
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [str(RUNTIME_BIN_DIR / "prank"), "predict", "-f", str(receptor_pdb), "-o", str(out_dir), "-visualizations", "0"]
    res = run_cmd(cmd, check=False)
    if res.returncode != 0:
        raise RuntimeError(f"P2Rank failed:\nSTDOUT:\n{res.stdout}\nSTDERR:\n{res.stderr}")
    pred_csv = find_p2rank_predictions_csv(out_dir)
    if pred_csv is None:
        raise FileNotFoundError("P2Rank predictions CSV not found.")
    df = pd.read_csv(pred_csv)
    pockets = []
    residue_col = next((c for c in df.columns if "residue" in c.lower()), None)
    cx_col = next((c for c in df.columns if c.lower() == "center_x"), None)
    cy_col = next((c for c in df.columns if c.lower() == "center_y"), None)
    cz_col = next((c for c in df.columns if c.lower() == "center_z"), None)
    name_col = next((c for c in df.columns if c.lower() == "name"), None)
    score_col = next((c for c in df.columns if c.lower() == "score"), None)
    prob_col = next((c for c in df.columns if "prob" in c.lower()), None)
    for idx, row in df.head(MAX_P2RANK_POCKETS).iterrows():
        residues = parse_residue_ids_from_text(row.get(residue_col, "")) if residue_col else set()
        pockets.append({
            "method": "p2rank",
            "rank": int(idx + 1),
            "name": str(row.get(name_col, f"p2rank_pocket_{idx+1}")),
            "score": float(row.get(score_col)) if score_col and pd.notna(row.get(score_col)) else None,
            "probability": float(row.get(prob_col)) if prob_col and pd.notna(row.get(prob_col)) else None,
            "center": [
                float(row.get(cx_col)) if cx_col and pd.notna(row.get(cx_col)) else None,
                float(row.get(cy_col)) if cy_col and pd.notna(row.get(cy_col)) else None,
                float(row.get(cz_col)) if cz_col and pd.notna(row.get(cz_col)) else None
            ],
            "residue_ids": sorted(list(residues)),
            "n_residues": len(residues),
            "source_file": str(pred_csv)
        })
    return {"method": "p2rank", "output_dir": str(out_dir), "predictions_csv": str(pred_csv), "pockets": pockets}

_last_dogsite_submit_time = 0.0
def maybe_sleep_before_dogsite():
    global _last_dogsite_submit_time
    now = time.time()
    elapsed = now - _last_dogsite_submit_time
    if elapsed < DOGSITE_INITIAL_BACKOFF_SEC:
        time.sleep(DOGSITE_INITIAL_BACKOFF_SEC - elapsed)
    _last_dogsite_submit_time = time.time()

def download_binary(url, out_path):
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    out_path.write_bytes(r.content)
    return out_path

def robust_dogsite_submit(pdb_code):
    wait_s = DOGSITE_INITIAL_BACKOFF_SEC
    for attempt in range(DOGSITE_MAX_RETRIES_429 + 1):
        maybe_sleep_before_dogsite()
        status, payload = dogsite_submit_pdbcode_v1(
            pdb_code=pdb_code,
            analysis_detail=DOGSITE_ANALYSIS_DETAIL,
            granularity=DOGSITE_GRANULARITY,
            ligand="",
            chain=""
        )
        if status in (200, 202):
            return status, payload
        if status == 429 and attempt < DOGSITE_MAX_RETRIES_429:
            time.sleep(wait_s)
            wait_s *= 2
            continue
        raise RuntimeError(f"DoGSite submission failed with status {status}: {payload}")
    raise RuntimeError("DoGSite submission failed after retries")

def robust_dogsite_poll(job_ref):
    wait_s = DOGSITE_INITIAL_BACKOFF_SEC
    last_payload = None
    last_status = None
    for _ in range(DOGSITE_MAX_POLLS):
        status, payload = dogsite_get_job_v1(job_ref)
        last_status = status
        last_payload = payload
        if status == 200:
            return status, payload
        if status == 202:
            time.sleep(DOGSITE_POLL_INTERVAL_SEC)
            continue
        if status == 429:
            time.sleep(wait_s)
            wait_s = min(wait_s * 2, 180)
            continue
        raise RuntimeError(f"DoGSite polling failed with status {status}: {payload}")
    raise TimeoutError(f"DoGSite did not finish in time. Last status={last_status}, payload={last_payload}")

def try_extract_upload_id(upload_response):
    if not isinstance(upload_response, dict):
        return None
    for k in ["id", "pdbCode", "pdb_id", "code"]:
        if isinstance(upload_response.get(k), str) and upload_response[k].strip():
            return upload_response[k].strip()
    stack = [upload_response]
    while stack:
        cur = stack.pop()
        if isinstance(cur, dict):
            stack.extend(cur.values())
        elif isinstance(cur, list):
            stack.extend(cur)
        elif isinstance(cur, str):
            s = cur.strip()
            if s and "http" not in s and " " not in s and len(s) >= 6:
                return s
    return None

def run_dogsite_by_pdb_code(pdb_code, target_safe_name):
    out_dir = STEP5R_METHODS_DIR / target_safe_name / "dogsite"
    out_dir.mkdir(parents=True, exist_ok=True)
    cache_json = DOGSITE_CACHE_DIR / f"{target_safe_name}_{pdb_code}_dogsite.json"
    if cache_json.exists() and not FORCE_RERUN_DOGSITE:
        with open(cache_json, "r", encoding="utf-8") as f:
            return json.load(f)
    submit_status, submit_data = robust_dogsite_submit(pdb_code)
    job_ref = submit_data.get("location") or submit_data.get("id") or submit_data
    final_status, final_payload = robust_dogsite_poll(job_ref)
    residues_urls = []
    if isinstance(final_payload, dict) and isinstance(final_payload.get("residues"), list):
        residues_urls = [x for x in final_payload["residues"] if isinstance(x, str)]
    pockets = []
    for idx, url in enumerate(residues_urls[:MAX_DOGSITE_POCKETS], start=1):
        out_pdb = out_dir / f"dogsite_residues_{idx}.pdb"
        try:
            download_binary(url, out_pdb)
        except Exception:
            continue
        structure = get_parser_for_path(out_pdb).get_structure(f"dogsite_{idx}", str(out_pdb))
        residue_ids = set()
        coords_all = []
        for model in structure:
            for chain in model:
                for residue in chain:
                    if residue.id[0] != " ":
                        continue
                    residue_ids.add(residue_key(chain.id, residue))
                    for atom in residue.get_atoms():
                        if atom_is_h(atom):
                            continue
                        coords_all.append(atom.coord)
        center = [None, None, None]
        if coords_all:
            arr = np.array(coords_all, dtype=float)
            ctr = arr.mean(axis=0)
            center = [float(ctr[0]), float(ctr[1]), float(ctr[2])]
        pockets.append({
            "method": "dogsite",
            "rank": idx,
            "name": f"dogsite_pocket_{idx}",
            "score": None,
            "probability": None,
            "center": center,
            "residue_ids": sorted(list(residue_ids)),
            "n_residues": len(residue_ids),
            "source_file": str(out_pdb),
            "source_url": url
        })
    result = {
        "method": "dogsite",
        "pdb_code_used": pdb_code,
        "submit_status": submit_status,
        "final_status": final_status,
        "job_payload_final": final_payload,
        "output_dir": str(out_dir),
        "pockets": pockets
    }
    with open(cache_json, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    return result

def run_dogsite_with_fallback_inputs(target_report, target_safe_name):
    selected_structure = target_report.get("selected_structure_from_step3") or {}
    pdb_id = selected_structure.get("pdb_id")
    if isinstance(pdb_id, str) and re.fullmatch(r"[A-Za-z0-9]{4}", pdb_id):
        return run_dogsite_by_pdb_code(pdb_id.upper(), target_safe_name)
    raw_file = selected_structure.get("file_path")
    if raw_file is None or not Path(raw_file).exists():
        raise FileNotFoundError("No valid raw structure file available for DoGSite fallback upload.")
    pdb_upload_dir = STEP5R_TMP_DIR / target_safe_name
    pdb_upload_dir.mkdir(parents=True, exist_ok=True)
    pdb_for_upload = normalize_structure_to_pdb(raw_file, pdb_upload_dir / f"{target_safe_name}_dogsite_upload.pdb")
    upload_status, upload_data = proteinsplus_upload_pdb_v1(pdb_for_upload)
    if upload_status not in (200, 202):
        raise RuntimeError(f"ProteinsPlus upload failed with status {upload_status}: {upload_data}")
    upload_id = try_extract_upload_id(upload_data)
    if upload_id is None:
        raise RuntimeError(f"Could not extract upload id from ProteinsPlus response: {upload_data}")
    return run_dogsite_by_pdb_code(upload_id, target_safe_name)

def best_two_method_consensus(p2rank_pockets, dogsite_pockets):
    p2rank_pockets = [p for p in p2rank_pockets if p.get("n_residues", 0) > 0]
    dogsite_pockets = [p for p in dogsite_pockets if p.get("n_residues", 0) > 0]
    if not p2rank_pockets or not dogsite_pockets:
        return None
    best = None
    for p in p2rank_pockets:
        for d in dogsite_pockets:
            s1 = set(p["residue_ids"]); s2 = set(d["residue_ids"])
            j = jaccard(s1, s2)
            inter = s1 & s2
            union = s1 | s2
            final_set = inter if inter else union
            ctype = "intersection" if inter else "union_fallback"
            cand = {
                "methods_used": ["p2rank", "dogsite"],
                "selected_pockets": [
                    {"method": p["method"], "rank": p["rank"], "name": p["name"], "n_residues": p["n_residues"], "source_file": p.get("source_file"), "center": p.get("center"), "residue_ids": p.get("residue_ids", [])},
                    {"method": d["method"], "rank": d["rank"], "name": d["name"], "n_residues": d["n_residues"], "source_file": d.get("source_file"), "center": d.get("center"), "residue_ids": d.get("residue_ids", [])}
                ],
                "pairwise_jaccard_mean": float(j),
                "pairwise_jaccard_values": [float(j)],
                "consensus_type": ctype,
                "consensus_residue_ids": sorted(list(final_set)),
                "n_consensus_residues": len(final_set),
                "rank_penalty": int(p["rank"]) + int(d["rank"])
            }
            if best is None:
                best = cand
            else:
                new_key = (cand["pairwise_jaccard_mean"], cand["n_consensus_residues"], -cand["rank_penalty"])
                old_key = (best["pairwise_jaccard_mean"], best["n_consensus_residues"], -best["rank_penalty"])
                if new_key > old_key:
                    best = cand
    return best

def top_p2rank_fallback(p2rank_result):
    pockets = [p for p in p2rank_result.get("pockets", []) if p.get("n_residues", 0) > 0]
    if not pockets:
        return None
    top = pockets[0]
    return {
        "methods_used": ["p2rank"],
        "selected_pockets": [{
            "method": top["method"], "rank": top["rank"], "name": top["name"],
            "n_residues": top["n_residues"], "source_file": top.get("source_file"),
            "center": top.get("center"), "residue_ids": top.get("residue_ids", [])
        }],
        "pairwise_jaccard_mean": None,
        "pairwise_jaccard_values": [],
        "consensus_type": "single_method_fallback_p2rank",
        "consensus_residue_ids": top.get("residue_ids", []),
        "n_consensus_residues": top.get("n_residues", 0),
        "rank_penalty": int(top["rank"])
    }

repaired_reports = []
for rr in targets_for_consensus:
    target_name = rr.get("target_input")
    safe_target = sanitize_name(target_name)
    receptor_pdb = (rr.get("preprocess_result") or {}).get("clean_receptor_pdb")
    report = {
        "target_input": target_name,
        "normalized_target": rr.get("normalized_target"),
        "timestamp": datetime.now().isoformat(),
        "receptor_pdb": receptor_pdb,
        "selected_structure_from_step3": rr.get("selected_structure_from_step3"),
        "methods": {},
        "consensus": None,
        "consensus_grid": None,
        "status": "started"
    }
    try:
        if receptor_pdb is None or not Path(receptor_pdb).exists():
            raise FileNotFoundError(f"Preprocessed receptor not found: {receptor_pdb}")
        p2rank_res = run_p2rank_on_receptor(receptor_pdb, safe_target)
        report["methods"]["p2rank"] = p2rank_res
        add_status(f"{target_name}: P2Rank", True, f"{len([p for p in p2rank_res['pockets'] if p.get('n_residues',0)>0])} nonempty pockets")
        dogsite_res = None
        try:
            dogsite_res = run_dogsite_with_fallback_inputs(rr, safe_target)
            report["methods"]["dogsite"] = dogsite_res
            add_status(f"{target_name}: DoGSite", True, f"{len([p for p in dogsite_res['pockets'] if p.get('n_residues',0)>0])} nonempty pockets")
        except Exception as e:
            report["methods"]["dogsite"] = {"error": f"{type(e).__name__}: {e}", "pockets": []}
            add_status(f"{target_name}: DoGSite", False, f"{type(e).__name__}: {e}")

        consensus = best_two_method_consensus(p2rank_res.get("pockets", []), dogsite_res.get("pockets", [])) if dogsite_res is not None else None
        if consensus is None and ALLOW_SINGLE_METHOD_FALLBACK:
            consensus = top_p2rank_fallback(p2rank_res)
        report["consensus"] = consensus

        if consensus is None:
            report["status"] = "no_consensus"
            add_status(f"{target_name}: repaired consensus", False, "No 2-method consensus and no fallback pocket available")
        else:
            grid = compute_grid_from_residue_ids(receptor_pdb, consensus["consensus_residue_ids"], padding=CONSENSUS_GRID_PADDING, min_box=CONSENSUS_MIN_BOX_SIZE)
            report["consensus_grid"] = grid
            if grid is None:
                report["status"] = "consensus_but_no_grid"
                add_status(f"{target_name}: repaired consensus grid", False, "Pocket chosen but grid could not be computed")
            else:
                report["status"] = "ok"
                add_status(f"{target_name}: repaired consensus", True, f"{consensus['consensus_type']} | methods={','.join(consensus['methods_used'])} | n_res={consensus['n_consensus_residues']}")
                grid_json_path = STEP5R_GRIDS_DIR / f"{safe_target}_grid.json"
                with open(grid_json_path, "w", encoding="utf-8") as f:
                    json.dump({"target": target_name, "receptor_pdb": receptor_pdb, "consensus": consensus, "grid": grid}, f, indent=2, ensure_ascii=False)
                report["consensus_grid_file"] = str(grid_json_path)
        report_path = STEP5R_REPORTS_DIR / f"{safe_target}_repaired_consensus_report.json"
        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2, ensure_ascii=False)
        report["report_path"] = str(report_path)
    except Exception as e:
        report["status"] = "error"
        report["error"] = f"{type(e).__name__}: {e}"
        add_status(f"{target_name}: processing", False, f"{type(e).__name__}: {e}")
    repaired_reports.append(report)

summary_rows = []
for rep in repaired_reports:
    cons = rep.get("consensus") or {}
    grid = rep.get("consensus_grid") or {}
    summary_rows.append({
        "target_input": rep.get("target_input"),
        "status": rep.get("status"),
        "methods_used": ",".join(cons.get("methods_used", [])),
        "consensus_type": cons.get("consensus_type"),
        "n_consensus_residues": cons.get("n_consensus_residues"),
        "pairwise_jaccard_mean": cons.get("pairwise_jaccard_mean"),
        "grid_center_x": grid.get("center_x"),
        "grid_center_y": grid.get("center_y"),
        "grid_center_z": grid.get("center_z"),
        "grid_size_x": grid.get("size_x"),
        "grid_size_y": grid.get("size_y"),
        "grid_size_z": grid.get("size_z"),
        "receptor_pdb": rep.get("receptor_pdb"),
        "consensus_grid_file": rep.get("consensus_grid_file")
    })

manifest_path = STEP5R_MANIFESTS_DIR / "pocket_consensus_repaired_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump({"timestamp": datetime.now().isoformat(), "repaired_reports": repaired_reports, "summary_rows": summary_rows, "status": STATUS}, f, indent=2, ensure_ascii=False)

summary_tsv_path = STEP5R_REPORTS_DIR / "pocket_consensus_repaired_summary.tsv"
with open(summary_tsv_path, "w", encoding="utf-8") as f:
    headers = [
        "target_input", "status", "methods_used", "consensus_type",
        "n_consensus_residues", "pairwise_jaccard_mean",
        "grid_center_x", "grid_center_y", "grid_center_z",
        "grid_size_x", "grid_size_y", "grid_size_z",
        "receptor_pdb", "consensus_grid_file"
    ]
    f.write("\t".join(headers) + "\n")
    for row in summary_rows:
        f.write("\t".join([str(row.get(h, "")) for h in headers]) + "\n")

add_status("Repaired consensus manifest saved", True, str(manifest_path))
add_status("Repaired consensus summary TSV saved", True, str(summary_tsv_path))

print("\nReady.")
print("Repaired consensus manifest:", manifest_path)
print("Repaired consensus summary TSV:", summary_tsv_path)
show_status_panel(STATUS, ok_text="REPAIRED CONSENSUS STEP OK", bad_text="REPAIRED CONSENSUS STEP NOT OK")

## Consensus map, Jaccard audit, and final residue-set explanation

This block does **not** change docking. It audits the repaired consensus step and writes transparent per-target reports showing:

- pockets detected by each method
- pairwise Jaccard overlap between pocket residue sets
- residue vote table
- amino-acid composition of the final consensus site
- the exact residues that were ultimately used to build the docking grid


In [ ]:
# @title 05C. Build consensus maps, Jaccard tables, residue vote tables, and final grid residue sets

PROJECT_ROOT_DEFAULT = "/content/drive/MyDrive/oncotarget_pipeline"  # @param {type:"string"}

PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", PROJECT_ROOT_DEFAULT))
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}")

CONSENSUS_MANIFEST = PROJECT_ROOT / "consensus_repaired" / "manifests" / "pocket_consensus_repaired_manifest.json"
if not CONSENSUS_MANIFEST.exists():
    raise FileNotFoundError(f"Consensus manifest not found: {CONSENSUS_MANIFEST}")

OUT_DIR = PROJECT_ROOT / "consensus_maps"
OUT_TABLES = OUT_DIR / "tables"
OUT_JSON = OUT_DIR / "json"
for p in [OUT_DIR, OUT_TABLES, OUT_JSON]:
    p.mkdir(parents=True, exist_ok=True)

STATUS = []
def add_status(name, ok, details=""):
    STATUS.append({"name": str(name), "ok": bool(ok), "details": str(details)})

def jaccard(a, b):
    a = set(a)
    b = set(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def normalize_residue_id(residue_id):
    parts = str(residue_id).split("_")
    if len(parts) == 2:
        chain, resseq = parts
        resname = ""
    elif len(parts) >= 3:
        chain, resseq = parts[0], parts[1]
        resname = parts[2]
    else:
        chain, resseq, resname = "?", str(residue_id), ""

    residue_uid = f"{chain}_{resseq}" if not resname else f"{chain}_{resseq}_{resname}"
    aa_type = resname if resname else "UNK"
    return {
        "chain": chain,
        "resseq": resseq,
        "resname": resname,
        "aa_type": aa_type,
        "residue_uid": residue_uid
    }

def save_tsv(rows, path, headers):
    with open(path, "w", encoding="utf-8") as f:
        f.write("\t".join(headers) + "\n")
        for row in rows:
            f.write("\t".join([str(row.get(h, "")) for h in headers]) + "\n")

with open(CONSENSUS_MANIFEST, "r", encoding="utf-8") as f:
    manifest = json.load(f)

reports = manifest.get("repaired_reports", [])
add_status("Consensus repaired manifest loaded", True, str(CONSENSUS_MANIFEST))
add_status("Targets in repaired consensus manifest", True, str(len(reports)))

all_target_summaries = []

for rep in reports:
    target = rep.get("target_input")
    status = rep.get("status")
    methods = rep.get("methods", {})
    consensus = rep.get("consensus") or {}
    grid = rep.get("consensus_grid") or {}

    if status != "ok":
        add_status(f"{target}: skipped", False, f"status={status}")
        continue

    pocket_rows = []
    pocket_sets = {}
    for method_name, method_payload in methods.items():
        pockets = method_payload.get("pockets", []) if isinstance(method_payload, dict) else []
        for pocket in pockets:
            pocket_name = f"{method_name}_{pocket.get('rank')}"
            residue_ids = pocket.get("residue_ids", [])
            pocket_sets[pocket_name] = set(residue_ids)
            pocket_rows.append({
                "target_input": target,
                "method": method_name,
                "pocket_name": pocket_name,
                "rank": pocket.get("rank"),
                "n_residues": pocket.get("n_residues"),
                "center_x": (pocket.get("center") or [None, None, None])[0],
                "center_y": (pocket.get("center") or [None, None, None])[1],
                "center_z": (pocket.get("center") or [None, None, None])[2],
                "source_file": pocket.get("source_file")
            })

    jaccard_rows = []
    pocket_names = sorted(pocket_sets.keys())
    for p1 in pocket_names:
        for p2 in pocket_names:
            jaccard_rows.append({
                "target_input": target,
                "pocket_1": p1,
                "pocket_2": p2,
                "jaccard": round(jaccard(pocket_sets[p1], pocket_sets[p2]), 6)
            })

    selected_pockets = consensus.get("selected_pockets", [])
    selected_names = []
    selected_sets = {}
    for sp in selected_pockets:
        sp_name = f"{sp.get('method')}_{sp.get('rank')}"
        selected_names.append(sp_name)
        selected_sets[sp_name] = set(sp.get("residue_ids", []))

    residue_vote_rows = []
    residue_counter = Counter()
    residue_to_sources = defaultdict(dict)

    for sp_name, residues in selected_sets.items():
        for rid in residues:
            residue_counter[rid] += 1
            residue_to_sources[rid][sp_name] = 1

    final_consensus_set = set(consensus.get("consensus_residue_ids", []))

    for rid in sorted(residue_counter.keys()):
        norm = normalize_residue_id(rid)
        row = {
            "target_input": target,
            "residue_uid": norm["residue_uid"],
            "chain": norm["chain"],
            "resseq": norm["resseq"],
            "aa_type": norm["aa_type"],
            "votes": residue_counter[rid],
            "final_consensus": 1 if rid in final_consensus_set else 0
        }
        for sp_name in selected_names:
            row[sp_name] = residue_to_sources[rid].get(sp_name, 0)
        residue_vote_rows.append(row)

    aa_counter = Counter()
    for rid in final_consensus_set:
        norm = normalize_residue_id(rid)
        aa_counter[norm["aa_type"]] += 1

    aa_rows = []
    for aa_type, count in sorted(aa_counter.items(), key=lambda x: (-x[1], x[0])):
        aa_rows.append({
            "target_input": target,
            "aa_type": aa_type,
            "count_in_final_consensus": count
        })

    final_summary = {
        "target_input": target,
        "status": status,
        "consensus_type": consensus.get("consensus_type"),
        "methods_used": consensus.get("methods_used", []),
        "selected_pockets": selected_pockets,
        "final_consensus_residue_ids": sorted(list(final_consensus_set)),
        "n_final_consensus_residues": len(final_consensus_set),
        "grid": grid
    }

    pocket_tsv = OUT_TABLES / f"{target}_pocket_summary.tsv"
    jaccard_tsv = OUT_TABLES / f"{target}_jaccard_matrix.tsv"
    votes_tsv = OUT_TABLES / f"{target}_residue_votes.tsv"
    aa_tsv = OUT_TABLES / f"{target}_aa_composition.tsv"
    final_json = OUT_JSON / f"{target}_consensus_map.json"

    save_tsv(
        pocket_rows,
        pocket_tsv,
        ["target_input", "method", "pocket_name", "rank", "n_residues", "center_x", "center_y", "center_z", "source_file"]
    )
    save_tsv(
        jaccard_rows,
        jaccard_tsv,
        ["target_input", "pocket_1", "pocket_2", "jaccard"]
    )
    vote_headers = ["target_input", "residue_uid", "chain", "resseq", "aa_type", "votes", "final_consensus"] + selected_names
    save_tsv(residue_vote_rows, votes_tsv, vote_headers)
    save_tsv(aa_rows, aa_tsv, ["target_input", "aa_type", "count_in_final_consensus"])

    with open(final_json, "w", encoding="utf-8") as f:
        json.dump({
            "target_input": target,
            "pocket_summary": pocket_rows,
            "jaccard_matrix": jaccard_rows,
            "residue_votes": residue_vote_rows,
            "aa_composition": aa_rows,
            "final_summary": final_summary
        }, f, indent=2, ensure_ascii=False)

    add_status(f"{target}: consensus map built", True, f"{len(final_consensus_set)} final residues for grid")

    all_target_summaries.append({
        "target_input": target,
        "consensus_type": consensus.get("consensus_type"),
        "methods_used": ",".join(consensus.get("methods_used", [])),
        "n_final_consensus_residues": len(final_consensus_set),
        "final_consensus_residue_ids": ", ".join(sorted(list(final_consensus_set))),
        "grid_center_x": grid.get("center_x"),
        "grid_center_y": grid.get("center_y"),
        "grid_center_z": grid.get("center_z"),
        "grid_size_x": grid.get("size_x"),
        "grid_size_y": grid.get("size_y"),
        "grid_size_z": grid.get("size_z")
    })

summary_tsv = OUT_TABLES / "consensus_global_summary.tsv"
save_tsv(
    all_target_summaries,
    summary_tsv,
    [
        "target_input",
        "consensus_type",
        "methods_used",
        "n_final_consensus_residues",
        "final_consensus_residue_ids",
        "grid_center_x",
        "grid_center_y",
        "grid_center_z",
        "grid_size_x",
        "grid_size_y",
        "grid_size_z"
    ]
)

summary_json = OUT_JSON / "consensus_global_summary.json"
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump({
        "timestamp": datetime.now().isoformat(),
        "targets": all_target_summaries,
        "status": STATUS
    }, f, indent=2, ensure_ascii=False)

add_status("Global consensus summary TSV saved", True, str(summary_tsv))
add_status("Global consensus summary JSON saved", True, str(summary_json))

print("\nReady.")
print("Consensus global summary TSV:", summary_tsv)
print("Consensus global summary JSON:", summary_json)
print("\nFinal residues used for grid:")
for row in all_target_summaries:
    print(f"- {row['target_input']}: {row['final_consensus_residue_ids']}")

show_status_panel(STATUS, ok_text="CONSENSUS MAPS BUILT", bad_text="CONSENSUS MAP BUILD HAS ERRORS")


## GNINA setup + ligand preparation + docking manifests

In [ ]:
# @title 06. GNINA install + ligand preparation + docking manifests (fixed)

GNINA_RELEASE = "latest"  # @param ["latest", "v1.3.2"]
FORCE_REINSTALL_GNINA = False  # @param {type:"boolean"}
MAX_RDKIT_EMBED_ATTEMPTS = 50  # @param {type:"integer"}
MMFF_MAX_ITERS = 500  # @param {type:"integer"}
FORCE_OVERWRITE_LIGANDS = True  # @param {type:"boolean"}

STATUS = []
def add_status(name, ok, details=""):
    STATUS.append({"name": str(name), "ok": bool(ok), "details": str(details)})

STEP6_DIR = DIRS["docking_setup"]
STEP6_MANIFESTS_DIR = STEP6_DIR / "manifests"
STEP6_LIGANDS_DIR = STEP6_DIR / "ligands"
STEP6_GNINA_DIR = STEP6_DIR / "gnina"
STEP6_ARCHIVES_DIR = STEP6_GNINA_DIR / "archives"
STEP6_REPORTS_DIR = STEP6_DIR / "reports"
RUNTIME_GNINA_DIR = Path("/content/oncotarget_runtime/gnina_runtime")
RUNTIME_GNINA_BIN = RUNTIME_BIN_DIR / "gnina"
for p in [STEP6_MANIFESTS_DIR, STEP6_LIGANDS_DIR, STEP6_GNINA_DIR, STEP6_ARCHIVES_DIR, STEP6_REPORTS_DIR, RUNTIME_GNINA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata_payload = json.load(f)
with open(DIRS["routing"] / "manifests" / "routing_manifest.json", "r", encoding="utf-8") as f:
    routing_manifest = json.load(f)
with open(DIRS["consensus_repaired"] / "manifests" / "pocket_consensus_repaired_manifest.json", "r", encoding="utf-8") as f:
    repaired_manifest = json.load(f)

targets = metadata_payload.get("targets", [])
ligands_ru = metadata_payload.get("ligands", [])
add_status("Loaded targets metadata", True, f"{len(targets)} targets")
add_status("Loaded ligands metadata", True, f"{len(ligands_ru)} ligands")

def download_file(url, output_path, timeout=300):
    output_path = Path(output_path)
    with requests.get(url, stream=True, timeout=timeout, allow_redirects=True) as r:
        r.raise_for_status()
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    return output_path

def extract_archive(archive_path, dest_dir):
    archive_path = Path(archive_path)
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    if archive_path.name.endswith(".tar.gz") or archive_path.name.endswith(".tgz"):
        with tarfile.open(archive_path, "r:gz") as tar:
            tar.extractall(dest_dir)
    elif archive_path.suffix.lower() == ".zip":
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(dest_dir)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")
    return dest_dir

def ensure_executable(path):
    path = Path(path)
    mode = path.stat().st_mode
    path.chmod(mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

def write_wrapper(wrapper_path, real_executable):
    wrapper_path = Path(wrapper_path)
    real_executable = Path(real_executable)
    script = f"""#!/usr/bin/env bash
set -e
exec "{real_executable}" "$@"
"""
    wrapper_path.write_text(script, encoding="utf-8")
    ensure_executable(wrapper_path)
    return wrapper_path

def github_release_json(repo, release="latest"):
    if release == "latest":
        url = f"https://api.github.com/repos/{repo}/releases/latest"
    else:
        tag = release if release.startswith("v") else f"v{release}"
        url = f"https://api.github.com/repos/{repo}/releases/tags/{tag}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.json()

def choose_gnina_asset(assets):
    preferred = [a for a in assets if "linux" in a["name"].lower() and "gnina" in a["name"].lower()]
    preferred = sorted(preferred, key=lambda a: (1 if "cuda12.8" in a["name"].lower() else 0, len(a["name"])))
    if preferred:
        return preferred[0]
    any_gnina = [a for a in assets if "gnina" in a["name"].lower()]
    if any_gnina:
        return any_gnina[0]
    raise RuntimeError("Could not choose GNINA asset from release assets")

def find_first_executable(root_dir, name):
    candidates = [c for c in Path(root_dir).rglob(name) if c.is_file()]
    if not candidates:
        return None
    return sorted(candidates, key=lambda x: len(str(x)))[0]

def install_gnina():
    release_json = github_release_json("gnina/gnina", release=GNINA_RELEASE)
    release_name = release_json.get("tag_name") or release_json.get("name") or "unknown_release"
    assets = release_json.get("assets", [])
    if not assets:
        raise RuntimeError("No assets found in GNINA release.")
    asset = choose_gnina_asset(assets)
    archive_path = STEP6_ARCHIVES_DIR / asset["name"]
    if FORCE_REINSTALL_GNINA and archive_path.exists():
        archive_path.unlink()
    if not archive_path.exists():
        download_file(asset["browser_download_url"], archive_path)
    local_install_dir = RUNTIME_GNINA_DIR / f"gnina_{release_name}"
    if local_install_dir.exists():
        shutil.rmtree(local_install_dir)
    local_install_dir.mkdir(parents=True, exist_ok=True)
    if archive_path.suffix.lower() in {".zip", ".gz"} or archive_path.name.endswith(".tar.gz") or archive_path.name.endswith(".tgz"):
        extract_archive(archive_path, local_install_dir)
        gnina_real = find_first_executable(local_install_dir, "gnina")
        if gnina_real is None:
            raise FileNotFoundError("GNINA executable not found after extraction")
    else:
        gnina_real = local_install_dir / "gnina"
        shutil.copy2(archive_path, gnina_real)
    ensure_executable(gnina_real)
    write_wrapper(RUNTIME_GNINA_BIN, gnina_real)
    version_res = run_cmd([str(RUNTIME_GNINA_BIN), "--help"], check=False)
    return {
        "release": release_name,
        "asset_name": asset["name"],
        "asset_url": asset["browser_download_url"],
        "archive_path": str(archive_path),
        "install_dir_local": str(local_install_dir),
        "gnina_real_local": str(gnina_real),
        "gnina_wrapper_local": str(RUNTIME_GNINA_BIN),
        "help_output_head": (version_res.stdout + "\n" + version_res.stderr).strip()[:1200]
    }

if FORCE_REINSTALL_GNINA or not RUNTIME_GNINA_BIN.exists():
    gnina_info = install_gnina()
else:
    version_res = run_cmd([str(RUNTIME_GNINA_BIN), "--help"], check=False)
    gnina_info = {
        "release": "existing_runtime_install",
        "gnina_wrapper_local": str(RUNTIME_GNINA_BIN),
        "help_output_head": (version_res.stdout + "\n" + version_res.stderr).strip()[:1200]
    }

add_status("GNINA installed / available", True, str(RUNTIME_GNINA_BIN))
add_status("GNINA help/version check", True, gnina_info["help_output_head"][:300])

def safe_ligand_dir_name(lig_ru, query_name):
    cand = sanitize_name(query_name)
    if cand and cand != "unknown":
        return cand
    cand = sanitize_name(lig_ru)
    if cand and cand != "unknown":
        return cand
    return "ligand_unknown"

def write_mol_to_sdf(mol, out_sdf):
    writer = Chem.SDWriter(str(out_sdf))
    writer.write(mol)
    writer.close()

def ensure_3d_and_minimize(mol, max_attempts=50, mmff_max_iters=500):
    mol = Chem.AddHs(mol, addCoords=True)
    has_3d = False
    if mol.GetNumConformers() > 0:
        has_3d = mol.GetConformer().Is3D()
    if not has_3d:
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        params.maxIterations = int(max_attempts)
        status = AllChem.EmbedMolecule(mol, params)
        if status != 0:
            raise RuntimeError("RDKit 3D embedding failed.")
    if AllChem.MMFFHasAllMoleculeParams(mol):
        AllChem.MMFFOptimizeMolecule(mol, maxIters=int(mmff_max_iters))
        ff_used = "MMFF"
    else:
        AllChem.UFFOptimizeMolecule(mol, maxIters=int(mmff_max_iters))
        ff_used = "UFF"
    return mol, ff_used

def pubchem_sdf_url(cid, record_type="3d"):
    return f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/record/SDF?record_type={record_type}"

def download_pubchem_sdf(cid, out_sdf):
    out_sdf = Path(out_sdf)
    for record_type in ["3d", "2d"]:
        url = pubchem_sdf_url(cid, record_type=record_type)
        r = requests.get(url, timeout=120, allow_redirects=True)
        if r.status_code == 200 and r.text.strip():
            raw_sdf = out_sdf.with_name(out_sdf.stem + f"_{record_type}_raw.sdf")
            raw_sdf.write_bytes(r.content)
            supplier = Chem.SDMolSupplier(str(raw_sdf), removeHs=False)
            mol = None
            for m in supplier:
                if m is not None:
                    mol = m
                    break
            if mol is None:
                raise RuntimeError(f"PubChem SDF downloaded but RDKit could not parse it for CID {cid}")
            mol, ff_used = ensure_3d_and_minimize(mol, max_attempts=MAX_RDKIT_EMBED_ATTEMPTS, mmff_max_iters=MMFF_MAX_ITERS)
            write_mol_to_sdf(mol, out_sdf)
            return {
                "pubchem_cid": cid,
                "record_type_downloaded": record_type,
                "raw_sdf": str(raw_sdf),
                "forcefield_used": ff_used,
                "n_atoms": mol.GetNumAtoms(),
                "n_heavy_atoms": mol.GetNumHeavyAtoms(),
                "formula_rdkit": rdMolDescriptors.CalcMolFormula(mol)
            }
    raise RuntimeError(f"Could not download PubChem SDF for CID {cid}")

def build_from_smiles(smiles, out_sdf):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise RuntimeError("RDKit failed to parse SMILES.")
    mol, ff_used = ensure_3d_and_minimize(mol, max_attempts=MAX_RDKIT_EMBED_ATTEMPTS, mmff_max_iters=MMFF_MAX_ITERS)
    write_mol_to_sdf(mol, out_sdf)
    return {
        "source_type": "manual_smiles",
        "forcefield_used": ff_used,
        "n_atoms": mol.GetNumAtoms(),
        "n_heavy_atoms": mol.GetNumHeavyAtoms(),
        "formula_rdkit": rdMolDescriptors.CalcMolFormula(mol)
    }

def convert_reference_pdb_to_sdf(ref_pdb_path, out_sdf_path):
    res = run_cmd(["obabel", str(ref_pdb_path), "-O", str(out_sdf_path), "-h"], check=False)
    if res.returncode != 0 or not Path(out_sdf_path).exists():
        raise RuntimeError(f"Open Babel failed converting reference ligand to SDF.\nSTDOUT:\n{res.stdout}\nSTDERR:\n{res.stderr}")
    return str(out_sdf_path)

PUBCHEM_CID_FALLBACK = {
    "маравирок": {"query_name": "Maraviroc", "cid": 3002977},
    "пексидартиниб": {"query_name": "Pexidartinib", "cid": 25151352},
    "нифуроксазид": {"query_name": "Nifuroxazide", "cid": 5337997},
    "пазопаниб": {"query_name": "Pazopanib", "cid": 10113978},
    "ивермектин": {"query_name": "Ivermectin B1a", "cid": 6321424}
}
MANUAL_STRUCTURE_FALLBACK = {
    "Q702": {"query_name": "Adrixetinib", "smiles": "CCCN1C=C(OCC(F)(F)F)C(=N1)C(=O)NC2=NC=C(OC3=CC=NC4=CC(OC)=C(OC)C=C34)C=C2"},
    "TM5614": {"query_name": "TM5614", "smiles": "c1(C(=O)O)c(ccc(c1)Cl)NC(=O)COCC(=O)Nc1cc(ccc1)c1occc1"}
}

ligand_reports = []
for lig_ru in ligands_ru:
    if lig_ru in PUBCHEM_CID_FALLBACK:
        query_name = PUBCHEM_CID_FALLBACK[lig_ru]["query_name"]
    elif lig_ru in MANUAL_STRUCTURE_FALLBACK:
        query_name = MANUAL_STRUCTURE_FALLBACK[lig_ru]["query_name"]
    else:
        query_name = lig_ru
    safe_lig = safe_ligand_dir_name(lig_ru, query_name)
    lig_dir = STEP6_LIGANDS_DIR / safe_lig
    lig_dir.mkdir(parents=True, exist_ok=True)
    if FORCE_OVERWRITE_LIGANDS:
        for p in lig_dir.glob("*"):
            if p.is_file():
                p.unlink()
    sdf_path = lig_dir / f"{safe_lig}.sdf"
    json_path = lig_dir / f"{safe_lig}_source.json"
    report = {
        "ligand_input_name": lig_ru,
        "query_name": query_name,
        "safe_dir_name": safe_lig,
        "timestamp": datetime.now().isoformat(),
        "status": "started"
    }
    try:
        if lig_ru in PUBCHEM_CID_FALLBACK:
            src = PUBCHEM_CID_FALLBACK[lig_ru]
            prep_info = download_pubchem_sdf(src["cid"], sdf_path)
            report.update({"status": "ok", "source_type": "pubchem_cid_fallback", "cid": src["cid"], "sdf_path": str(sdf_path), "prep_info": prep_info})
        elif lig_ru in MANUAL_STRUCTURE_FALLBACK:
            src = MANUAL_STRUCTURE_FALLBACK[lig_ru]
            prep_info = build_from_smiles(src["smiles"], sdf_path)
            report.update({"status": "ok", "source_type": "manual_structure_fallback", "cid": None, "sdf_path": str(sdf_path), "prep_info": prep_info})
        else:
            raise RuntimeError("No curated fallback defined for this ligand.")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2, ensure_ascii=False)
        add_status(f"Ligand prepared: {lig_ru}", True, f"{report['source_type']} | {sdf_path}")
    except Exception as e:
        report.update({"status": "error", "error": f"{type(e).__name__}: {e}"})
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2, ensure_ascii=False)
        add_status(f"Ligand prepared: {lig_ru}", False, f"{type(e).__name__}: {e}")
    ligand_reports.append(report)

ligand_ok_map = {r["ligand_input_name"]: r for r in ligand_reports if r.get("status") == "ok"}

routing_reports = routing_manifest.get("routing_reports", [])
repaired_reports = repaired_manifest.get("repaired_reports", [])
docking_jobs = []
redocking_jobs = []

for rep in routing_reports:
    if rep.get("routing_decision") != "reference_ligand_first" or rep.get("status") != "ok":
        continue
    target = rep.get("target_input")
    safe_target = sanitize_name(target)
    receptor_pdb = (rep.get("preprocess_result") or {}).get("clean_receptor_pdb")
    ref_lig_pdb = rep.get("reference_ligand_pdb")
    grid = rep.get("reference_ligand_grid") or {}
    if not receptor_pdb or not Path(receptor_pdb).exists():
        add_status(f"Reference route receptor: {target}", False, "Missing preprocessed receptor")
        continue
    if not ref_lig_pdb or not Path(ref_lig_pdb).exists():
        add_status(f"Reference route ligand: {target}", False, "Missing extracted reference ligand PDB")
        continue
    add_status(f"Reference route ready: {target}", True, str(receptor_pdb))
    ref_dir = STEP6_LIGANDS_DIR / f"{safe_target}_reference"
    ref_dir.mkdir(parents=True, exist_ok=True)
    ref_sdf_path = ref_dir / f"{safe_target}_reference.sdf"
    try:
        convert_reference_pdb_to_sdf(ref_lig_pdb, ref_sdf_path)
        redocking_jobs.append({
            "target_input": target,
            "route_type": "reference_ligand_first",
            "receptor_pdb": receptor_pdb,
            "reference_ligand_pdb": ref_lig_pdb,
            "reference_ligand_sdf": str(ref_sdf_path),
            "grid": grid,
            "notes": "Run redocking and compare with crystal/reference pose."
        })
        add_status(f"Reference ligand converted: {target}", True, str(ref_sdf_path))
    except Exception as e:
        add_status(f"Reference ligand converted: {target}", False, f"{type(e).__name__}: {e}")

    for lig_ru, lig_rep in ligand_ok_map.items():
        docking_jobs.append({
            "target_input": target,
            "route_type": "reference_ligand_first",
            "ligand_input_name": lig_ru,
            "ligand_query_name": lig_rep.get("query_name"),
            "ligand_cid": lig_rep.get("cid"),
            "ligand_sdf": lig_rep["sdf_path"],
            "receptor_pdb": receptor_pdb,
            "grid": grid,
            "reference_ligand_pdb": ref_lig_pdb,
            "reference_ligand_grid_source": "routing_manifest"
        })

for rep in repaired_reports:
    if rep.get("status") != "ok":
        continue
    target = rep.get("target_input")
    receptor_pdb = rep.get("receptor_pdb")
    grid = rep.get("consensus_grid") or {}
    if not receptor_pdb or not Path(receptor_pdb).exists():
        add_status(f"Consensus route receptor: {target}", False, "Missing receptor PDB")
        continue
    if not grid:
        add_status(f"Consensus route grid: {target}", False, "Missing consensus grid")
        continue
    add_status(f"Consensus route ready: {target}", True, str(receptor_pdb))
    for lig_ru, lig_rep in ligand_ok_map.items():
        docking_jobs.append({
            "target_input": target,
            "route_type": rep.get("consensus", {}).get("consensus_type", "consensus"),
            "ligand_input_name": lig_ru,
            "ligand_query_name": lig_rep.get("query_name"),
            "ligand_cid": lig_rep.get("cid"),
            "ligand_sdf": lig_rep["sdf_path"],
            "receptor_pdb": receptor_pdb,
            "grid": grid,
            "reference_ligand_pdb": None,
            "reference_ligand_grid_source": "consensus_repaired_manifest"
        })

ligand_manifest_path = STEP6_MANIFESTS_DIR / "ligand_preparation_manifest_repaired_v2.json"
docking_manifest_path = STEP6_MANIFESTS_DIR / "docking_jobs_manifest_repaired_v2.json"
redocking_manifest_path = STEP6_MANIFESTS_DIR / "redocking_jobs_manifest_repaired_v2.json"

with open(ligand_manifest_path, "w", encoding="utf-8") as f:
    json.dump({"timestamp": datetime.now().isoformat(), "ligand_reports": ligand_reports, "status": STATUS}, f, indent=2, ensure_ascii=False)
with open(docking_manifest_path, "w", encoding="utf-8") as f:
    json.dump({"timestamp": datetime.now().isoformat(), "gnina_binary": str(RUNTIME_GNINA_BIN), "docking_jobs": docking_jobs, "redocking_jobs": redocking_jobs, "status": STATUS}, f, indent=2, ensure_ascii=False)
with open(redocking_manifest_path, "w", encoding="utf-8") as f:
    json.dump({"timestamp": datetime.now().isoformat(), "gnina_binary": str(RUNTIME_GNINA_BIN), "redocking_jobs": redocking_jobs, "status": STATUS}, f, indent=2, ensure_ascii=False)

jobs_tsv_path = STEP6_REPORTS_DIR / "docking_jobs_summary_repaired_v2.tsv"
with open(jobs_tsv_path, "w", encoding="utf-8") as f:
    headers = ["target_input", "route_type", "ligand_input_name", "ligand_query_name", "ligand_cid", "ligand_sdf", "receptor_pdb"]
    f.write("\t".join(headers) + "\n")
    for row in docking_jobs:
        f.write("\t".join([str(row.get(h, "")) for h in headers]) + "\n")

add_status("Ligand preparation manifest saved", True, str(ligand_manifest_path))
add_status("Docking jobs manifest saved", True, str(docking_manifest_path))
add_status("Redocking jobs manifest saved", True, str(redocking_manifest_path))
add_status("Docking jobs TSV saved", True, str(jobs_tsv_path))

print("\nReady.")
print("GNINA binary:", RUNTIME_GNINA_BIN)
print("Ligand manifest:", ligand_manifest_path)
print("Docking jobs manifest:", docking_manifest_path)
print("Redocking jobs manifest:", redocking_manifest_path)
print("Docking jobs TSV:", jobs_tsv_path)
show_status_panel(STATUS, ok_text="GNINA / LIGAND PREP STEP OK", bad_text="GNINA / LIGAND PREP STEP NOT OK")

## Redocking calibration

In [ ]:
# @title 07. Redocking calibration with GNINA + build final docking manifest

RUN_REDOCKING = True  # @param {type:"boolean"}
BUILD_FINAL_DOCKING_MANIFEST = True  # @param {type:"boolean"}

BOX_SCALE_FACTORS = "0.90,1.00,1.10"  # @param {type:"string"}
EXHAUSTIVENESS_VALUES = "8,16"  # @param {type:"string"}

NUM_MODES = 9  # @param {type:"integer"}
MIN_RMSD_FILTER = 1.0  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}
CPU = 2  # @param {type:"integer"}

CNN_SCORING = "rescore"  # @param ["none","rescore","refinement","metrorescore","metrorefine","all"]
POSE_SORT_ORDER = "CNNscore"  # @param ["CNNscore","CNNaffinity","Energy"]

MIN_SUCCESS_RMSD = 2.0  # @param {type:"number"}
USE_NO_GPU = False  # @param {type:"boolean"}

STATUS = []
def add_status(name, ok, details=""):
    STATUS.append({"name": str(name), "ok": bool(ok), "details": str(details)})

GNINA_BIN = RUNTIME_GNINA_BIN
STEP6_MANIFESTS_DIR = DIRS["docking_setup"] / "manifests"
redocking_manifest_path = STEP6_MANIFESTS_DIR / "redocking_jobs_manifest_repaired_v2.json"
docking_jobs_manifest_path = STEP6_MANIFESTS_DIR / "docking_jobs_manifest_repaired_v2.json"

with open(redocking_manifest_path, "r", encoding="utf-8") as f:
    redocking_manifest = json.load(f)
with open(docking_jobs_manifest_path, "r", encoding="utf-8") as f:
    docking_jobs_manifest = json.load(f)

redocking_jobs = redocking_manifest.get("redocking_jobs", [])
docking_jobs = docking_jobs_manifest.get("docking_jobs", [])
box_scales = [float(x.strip()) for x in BOX_SCALE_FACTORS.split(",") if x.strip()]
exhaustiveness_values = [int(x.strip()) for x in EXHAUSTIVENESS_VALUES.split(",") if x.strip()]

STEP7_DIR = DIRS["redocking_calibration"]
STEP7_RUNS_DIR = STEP7_DIR / "runs"
STEP7_REPORTS_DIR = STEP7_DIR / "reports"
STEP7_MANIFESTS_DIR = STEP7_DIR / "manifests"
for p in [STEP7_RUNS_DIR, STEP7_REPORTS_DIR, STEP7_MANIFESTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

add_status("Loaded redocking jobs manifest", True, str(redocking_manifest_path))
add_status("Loaded docking jobs manifest", True, str(docking_jobs_manifest_path))
add_status("Redocking jobs loaded", True, str(len(redocking_jobs)))
add_status("Docking jobs loaded", True, str(len(docking_jobs)))

def load_first_sdf_mol(path):
    suppl = Chem.SDMolSupplier(str(path), removeHs=False)
    for mol in suppl:
        if mol is not None:
            return mol
    raise RuntimeError(f"Could not parse any molecule from SDF: {path}")

def heavy_atom_best_rmsd(ref_sdf, probe_sdf):
    ref = Chem.RemoveHs(load_first_sdf_mol(ref_sdf))
    prb = Chem.RemoveHs(load_first_sdf_mol(probe_sdf))
    if ref.GetNumAtoms() != prb.GetNumAtoms():
        raise RuntimeError(f"Reference and probe atom counts differ after H removal: {ref.GetNumAtoms()} vs {prb.GetNumAtoms()}")
    return float(rdMolAlign.GetBestRMS(prb, ref))

def read_top_pose_scores(out_sdf):
    mol = load_first_sdf_mol(out_sdf)
    props = {}
    for name in mol.GetPropNames():
        try:
            props[name] = mol.GetProp(name)
        except Exception:
            pass
    return props

def scale_grid(grid, factor):
    return {
        "center_x": float(grid["center_x"]),
        "center_y": float(grid["center_y"]),
        "center_z": float(grid["center_z"]),
        "size_x": float(grid["size_x"]) * float(factor),
        "size_y": float(grid["size_y"]) * float(factor),
        "size_z": float(grid["size_z"]) * float(factor)
    }

def gnina_redock_once(job, grid, exhaustiveness, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_sdf = out_dir / "redocked.sdf"
    log_txt = out_dir / "gnina.log"
    cmd = [
        str(GNINA_BIN),
        "-r", str(job["receptor_pdb"]),
        "-l", str(job["reference_ligand_sdf"]),
        "--center_x", str(grid["center_x"]),
        "--center_y", str(grid["center_y"]),
        "--center_z", str(grid["center_z"]),
        "--size_x", str(grid["size_x"]),
        "--size_y", str(grid["size_y"]),
        "--size_z", str(grid["size_z"]),
        "--cnn_scoring", str(CNN_SCORING),
        "--pose_sort_order", str(POSE_SORT_ORDER),
        "--exhaustiveness", str(exhaustiveness),
        "--num_modes", str(NUM_MODES),
        "--min_rmsd_filter", str(MIN_RMSD_FILTER),
        "--cpu", str(CPU),
        "--seed", str(SEED),
        "--out", str(out_sdf),
        "--log", str(log_txt)
    ]
    if USE_NO_GPU:
        cmd.append("--no_gpu")
    res = run_cmd(cmd, check=False)
    if res.returncode != 0:
        raise RuntimeError(f"GNINA failed.\nSTDOUT:\n{res.stdout}\nSTDERR:\n{res.stderr}")
    if not out_sdf.exists():
        raise FileNotFoundError(f"GNINA finished but output SDF was not found: {out_sdf}")
    scores = read_top_pose_scores(out_sdf)
    rmsd = heavy_atom_best_rmsd(job["reference_ligand_sdf"], out_sdf)
    return {"out_sdf": str(out_sdf), "log_txt": str(log_txt), "scores": scores, "rmsd": rmsd}

redocking_reports = []
best_grid_by_target = {}

if RUN_REDOCKING:
    for job in redocking_jobs:
        target = job["target_input"]
        safe_target = sanitize_name(target)
        target_dir = STEP7_RUNS_DIR / safe_target
        target_dir.mkdir(parents=True, exist_ok=True)
        job_report = {
            "target_input": target,
            "timestamp": datetime.now().isoformat(),
            "receptor_pdb": job["receptor_pdb"],
            "reference_ligand_sdf": job["reference_ligand_sdf"],
            "base_grid": job["grid"],
            "sweep_results": [],
            "best_result": None,
            "status": "started"
        }
        try:
            for scale in box_scales:
                scaled_grid = scale_grid(job["grid"], scale)
                for ex in exhaustiveness_values:
                    run_name = f"scale_{scale:.2f}_exh_{ex}"
                    run_dir = target_dir / run_name
                    try:
                        result = gnina_redock_once(job, scaled_grid, ex, run_dir)
                        job_report["sweep_results"].append({
                            "scale_factor": scale,
                            "exhaustiveness": ex,
                            "grid": scaled_grid,
                            "rmsd": result["rmsd"],
                            "out_sdf": result["out_sdf"],
                            "log_txt": result["log_txt"],
                            "scores": result["scores"]
                        })
                    except Exception as e:
                        job_report["sweep_results"].append({
                            "scale_factor": scale,
                            "exhaustiveness": ex,
                            "grid": scaled_grid,
                            "error": f"{type(e).__name__}: {e}"
                        })
            successful = [x for x in job_report["sweep_results"] if "rmsd" in x]
            if not successful:
                job_report["status"] = "failed"
                add_status(f"{target}: redocking sweep", False, "No successful GNINA run in sweep")
            else:
                successful = sorted(successful, key=lambda x: (x["rmsd"], -float(x["exhaustiveness"]), abs(x["scale_factor"] - 1.0)))
                best = successful[0]
                job_report["best_result"] = best
                best_grid_by_target[target] = best["grid"]
                if best["rmsd"] <= MIN_SUCCESS_RMSD:
                    job_report["status"] = "ok"
                    add_status(f"{target}: redocking best", True, f"RMSD={best['rmsd']:.3f} Å | scale={best['scale_factor']:.2f} | exh={best['exhaustiveness']}")
                else:
                    job_report["status"] = "best_above_threshold"
                    add_status(f"{target}: redocking best", False, f"Best RMSD={best['rmsd']:.3f} Å > {MIN_SUCCESS_RMSD:.1f} Å")
            report_path = STEP7_REPORTS_DIR / f"{safe_target}_redocking_report.json"
            with open(report_path, "w", encoding="utf-8") as f:
                json.dump(job_report, f, indent=2, ensure_ascii=False)
            job_report["report_path"] = str(report_path)
        except Exception as e:
            job_report["status"] = "error"
            job_report["error"] = f"{type(e).__name__}: {e}"
            add_status(f"{target}: redocking sweep", False, f"{type(e).__name__}: {e}")
        redocking_reports.append(job_report)

final_docking_jobs = []
if BUILD_FINAL_DOCKING_MANIFEST:
    for job in docking_jobs:
        target = job["target_input"]
        route_type = job.get("route_type", "")
        new_job = dict(job)
        if route_type == "reference_ligand_first" and target in best_grid_by_target:
            new_job["grid"] = best_grid_by_target[target]
            new_job["grid_source"] = "redocking_calibrated"
        else:
            new_job["grid_source"] = job.get("reference_ligand_grid_source", "precomputed")
        final_docking_jobs.append(new_job)
    add_status("Final docking jobs rebuilt", True, str(len(final_docking_jobs)))

redocking_summary_rows = []
for rep in redocking_reports:
    best = rep.get("best_result") or {}
    redocking_summary_rows.append({
        "target_input": rep.get("target_input"),
        "status": rep.get("status"),
        "best_rmsd": best.get("rmsd"),
        "best_scale_factor": best.get("scale_factor"),
        "best_exhaustiveness": best.get("exhaustiveness"),
        "best_out_sdf": best.get("out_sdf"),
        "best_log_txt": best.get("log_txt")
    })

calibration_manifest_path = STEP7_MANIFESTS_DIR / "redocking_calibration_manifest.json"
with open(calibration_manifest_path, "w", encoding="utf-8") as f:
    json.dump({
        "timestamp": datetime.now().isoformat(),
        "redocking_reports": redocking_reports,
        "redocking_summary_rows": redocking_summary_rows,
        "status": STATUS
    }, f, indent=2, ensure_ascii=False)

summary_tsv_path = STEP7_REPORTS_DIR / "redocking_calibration_summary.tsv"
with open(summary_tsv_path, "w", encoding="utf-8") as f:
    headers = ["target_input", "status", "best_rmsd", "best_scale_factor", "best_exhaustiveness", "best_out_sdf", "best_log_txt"]
    f.write("\t".join(headers) + "\n")
    for row in redocking_summary_rows:
        f.write("\t".join([str(row.get(h, "")) for h in headers]) + "\n")

add_status("Redocking calibration manifest saved", True, str(calibration_manifest_path))
add_status("Redocking calibration summary TSV saved", True, str(summary_tsv_path))

if BUILD_FINAL_DOCKING_MANIFEST:
    final_manifest_path = STEP7_MANIFESTS_DIR / "final_docking_jobs_manifest.json"
    with open(final_manifest_path, "w", encoding="utf-8") as f:
        json.dump({
            "timestamp": datetime.now().isoformat(),
            "gnina_binary": str(GNINA_BIN),
            "best_grid_by_target": best_grid_by_target,
            "docking_jobs": final_docking_jobs,
            "status": STATUS
        }, f, indent=2, ensure_ascii=False)
    add_status("Final docking jobs manifest saved", True, str(final_manifest_path))

print("\nReady.")
print("Redocking calibration manifest:", calibration_manifest_path)
print("Redocking summary TSV:", summary_tsv_path)
if BUILD_FINAL_DOCKING_MANIFEST:
    print("Final docking jobs manifest:", final_manifest_path)
show_status_panel(STATUS, ok_text="REDOCKING CALIBRATION OK", bad_text="REDOCKING CALIBRATION NOT OK")

## Production docking (batched)

In [ ]:
# @title 08. Run GNINA docking jobs in batches + resume support

JOB_INDEX_START = 0  # @param {type:"integer"}
JOB_INDEX_END = 10  # @param {type:"integer"}
FILTER_TARGETS_CSV = ""  # @param {type:"string"}
FILTER_LIGANDS_CSV = ""  # @param {type:"string"}
RESUME_SKIP_FINISHED = True  # @param {type:"boolean"}

EXHAUSTIVENESS_DOCK = 8  # @param {type:"integer"}
NUM_MODES_DOCK = 12  # @param {type:"integer"}
MIN_RMSD_FILTER_DOCK = 1.0  # @param {type:"number"}
CPU_DOCK = 2  # @param {type:"integer"}
SEED_DOCK = 42  # @param {type:"integer"}

CNN_SCORING_DOCK = "rescore"  # @param ["none","rescore","refinement","metrorescore","metrorefine","all"]
POSE_SORT_ORDER_DOCK = "CNNscore"  # @param ["CNNscore","CNNaffinity","Energy"]
USE_NO_GPU_DOCK = False  # @param {type:"boolean"}

STATUS = []
def add_status(name, ok, details=""):
    STATUS.append({"name": str(name), "ok": bool(ok), "details": str(details)})

GNINA_BIN = RUNTIME_GNINA_BIN
final_manifest_path = DIRS["redocking_calibration"] / "manifests" / "final_docking_jobs_manifest.json"
with open(final_manifest_path, "r", encoding="utf-8") as f:
    docking_manifest = json.load(f)

STEP8_DIR = DIRS["docking_runs"]
STEP8_RUNS_DIR = STEP8_DIR / "runs"
STEP8_REPORTS_DIR = STEP8_DIR / "reports"
STEP8_MANIFESTS_DIR = STEP8_DIR / "manifests"
for p in [STEP8_RUNS_DIR, STEP8_REPORTS_DIR, STEP8_MANIFESTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def load_first_sdf_mol(path):
    suppl = Chem.SDMolSupplier(str(path), removeHs=False)
    for mol in suppl:
        if mol is not None:
            return mol
    raise RuntimeError(f"Could not parse any molecule from SDF: {path}")

def extract_top_pose_props(out_sdf):
    mol = load_first_sdf_mol(out_sdf)
    props = {}
    for name in mol.GetPropNames():
        try:
            props[name] = mol.GetProp(name)
        except Exception:
            pass
    return props

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

def run_gnina_docking(job, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_sdf = out_dir / "docked.sdf"
    log_txt = out_dir / "gnina.log"
    grid = job["grid"]
    cmd = [
        str(GNINA_BIN),
        "-r", str(job["receptor_pdb"]),
        "-l", str(job["ligand_sdf"]),
        "--center_x", str(grid["center_x"]),
        "--center_y", str(grid["center_y"]),
        "--center_z", str(grid["center_z"]),
        "--size_x", str(grid["size_x"]),
        "--size_y", str(grid["size_y"]),
        "--size_z", str(grid["size_z"]),
        "--cnn_scoring", str(CNN_SCORING_DOCK),
        "--pose_sort_order", str(POSE_SORT_ORDER_DOCK),
        "--exhaustiveness", str(EXHAUSTIVENESS_DOCK),
        "--num_modes", str(NUM_MODES_DOCK),
        "--min_rmsd_filter", str(MIN_RMSD_FILTER_DOCK),
        "--cpu", str(CPU_DOCK),
        "--seed", str(SEED_DOCK),
        "--out", str(out_sdf),
        "--log", str(log_txt)
    ]
    if USE_NO_GPU_DOCK:
        cmd.append("--no_gpu")
    res = run_cmd(cmd, check=False)
    if res.returncode != 0:
        raise RuntimeError(f"GNINA docking failed.\nSTDOUT:\n{res.stdout}\nSTDERR:\n{res.stderr}")
    if not out_sdf.exists():
        raise FileNotFoundError(f"GNINA finished but output SDF not found: {out_sdf}")
    return {"out_sdf": str(out_sdf), "log_txt": str(log_txt), "top_pose_props": extract_top_pose_props(out_sdf)}

all_jobs = docking_manifest.get("docking_jobs", [])
target_filters = set(split_csv(FILTER_TARGETS_CSV))
ligand_filters = set(split_csv(FILTER_LIGANDS_CSV))
filtered_jobs = []
for idx, job in enumerate(all_jobs):
    target = job.get("target_input", "")
    ligand = job.get("ligand_input_name", "")
    if target_filters and target not in target_filters:
        continue
    if ligand_filters and ligand not in ligand_filters:
        continue
    job2 = dict(job)
    job2["_global_index"] = idx
    filtered_jobs.append(job2)

selected_jobs = filtered_jobs[int(JOB_INDEX_START):] if int(JOB_INDEX_END) <= 0 else filtered_jobs[int(JOB_INDEX_START):int(JOB_INDEX_END)]

add_status("Loaded docking jobs manifest", True, str(final_manifest_path))
add_status("Total jobs in manifest", True, str(len(all_jobs)))
add_status("Jobs after filters", True, str(len(filtered_jobs)))
add_status("Jobs selected for this batch", True, str(len(selected_jobs)))

batch_reports = []
for local_idx, job in enumerate(selected_jobs, start=1):
    global_idx = job["_global_index"]
    target = job.get("target_input")
    ligand = job.get("ligand_input_name")
    route = job.get("route_type", "unknown")
    safe_target = sanitize_name(target)
    safe_ligand = sanitize_name(job.get("ligand_query_name") or ligand)

    run_dir = STEP8_RUNS_DIR / f"{global_idx:04d}_{safe_target}__{safe_ligand}"
    report_json = STEP8_REPORTS_DIR / f"{global_idx:04d}_{safe_target}__{safe_ligand}.json"
    out_sdf = run_dir / "docked.sdf"

    report = {
        "timestamp": datetime.now().isoformat(),
        "global_job_index": global_idx,
        "batch_position": local_idx,
        "target_input": target,
        "ligand_input_name": ligand,
        "ligand_query_name": job.get("ligand_query_name"),
        "route_type": route,
        "receptor_pdb": job.get("receptor_pdb"),
        "ligand_sdf": job.get("ligand_sdf"),
        "grid": job.get("grid"),
        "status": "started"
    }
    try:
        if RESUME_SKIP_FINISHED and report_json.exists() and out_sdf.exists():
            with open(report_json, "r", encoding="utf-8") as f:
                old = json.load(f)
            batch_reports.append(old)
            add_status(f"[{global_idx}] {target} / {ligand}", True, "Skipped (already finished)")
            continue
        result = run_gnina_docking(job, run_dir)
        props = result["top_pose_props"]
        report.update({
            "status": "ok",
            "out_sdf": result["out_sdf"],
            "log_txt": result["log_txt"],
            "top_pose_props": props,
            "CNNscore": safe_float(props.get("CNNscore")),
            "CNNaffinity": safe_float(props.get("CNNaffinity")),
            "minimizedAffinity": safe_float(props.get("minimizedAffinity")),
        })
        with open(report_json, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2, ensure_ascii=False)
        batch_reports.append(report)
        add_status(f"[{global_idx}] {target} / {ligand}", True, f"CNNscore={report.get('CNNscore')} | CNNaffinity={report.get('CNNaffinity')} | Energy={report.get('minimizedAffinity')}")
    except Exception as e:
        report.update({"status": "error", "error": f"{type(e).__name__}: {e}"})
        with open(report_json, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2, ensure_ascii=False)
        batch_reports.append(report)
        add_status(f"[{global_idx}] {target} / {ligand}", False, f"{type(e).__name__}: {e}")

summary_rows = []
for rep in batch_reports:
    summary_rows.append({
        "global_job_index": rep.get("global_job_index"),
        "target_input": rep.get("target_input"),
        "ligand_input_name": rep.get("ligand_input_name"),
        "ligand_query_name": rep.get("ligand_query_name"),
        "route_type": rep.get("route_type"),
        "status": rep.get("status"),
        "CNNscore": rep.get("CNNscore"),
        "CNNaffinity": rep.get("CNNaffinity"),
        "minimizedAffinity": rep.get("minimizedAffinity"),
        "out_sdf": rep.get("out_sdf"),
        "log_txt": rep.get("log_txt")
    })

batch_tag = f"{int(JOB_INDEX_START):04d}_{(int(JOB_INDEX_END) if int(JOB_INDEX_END) > 0 else len(filtered_jobs)):04d}"
batch_manifest_path = STEP8_MANIFESTS_DIR / f"docking_batch_{batch_tag}.json"
with open(batch_manifest_path, "w", encoding="utf-8") as f:
    json.dump({"timestamp": datetime.now().isoformat(), "summary_rows": summary_rows, "status": STATUS}, f, indent=2, ensure_ascii=False)

batch_tsv_path = STEP8_REPORTS_DIR / f"docking_batch_{batch_tag}.tsv"
with open(batch_tsv_path, "w", encoding="utf-8") as f:
    headers = ["global_job_index", "target_input", "ligand_input_name", "ligand_query_name", "route_type", "status", "CNNscore", "CNNaffinity", "minimizedAffinity", "out_sdf", "log_txt"]
    f.write("\t".join(headers) + "\n")
    for row in summary_rows:
        f.write("\t".join([str(row.get(h, "")) for h in headers]) + "\n")

add_status("Docking batch manifest saved", True, str(batch_manifest_path))
add_status("Docking batch TSV saved", True, str(batch_tsv_path))

print("\nReady.")
print("Source jobs manifest:", final_manifest_path)
print("Batch manifest:", batch_manifest_path)
print("Batch TSV:", batch_tsv_path)
show_status_panel(STATUS, ok_text="DOCKING BATCH OK", bad_text="DOCKING BATCH HAS ERRORS")

## Auto patch for missing FCGR3A jobs

In [ ]:
# @title 09. Auto-rerun missing FCGR3A jobs (detect missing ligands automatically)

PROJECT_ROOT_DEFAULT = "/content/drive/MyDrive/oncotarget_pipeline"  # @param {type:"string"}
TARGET_NAME = "FCGR3A"  # @param {type:"string"}
ONLY_IF_MISSING = True  # @param {type:"boolean"}
MAX_JOBS_TO_RUN = 10  # @param {type:"integer"}

EXHAUSTIVENESS_DOCK = 8  # @param {type:"integer"}
NUM_MODES_DOCK = 12  # @param {type:"integer"}
MIN_RMSD_FILTER_DOCK = 1.0  # @param {type:"number"}
CPU_DOCK = 2  # @param {type:"integer"}
SEED_DOCK = 42  # @param {type:"integer"}

CNN_SCORING_DOCK = "rescore"  # @param ["none","rescore","refinement","metrorescore","metrorefine","all"]
POSE_SORT_ORDER_DOCK = "CNNscore"  # @param ["CNNscore","CNNaffinity","Energy"]
USE_NO_GPU_DOCK = False  # @param {type:"boolean"}

STATUS = []
def add_status(name, ok, details=""):
    STATUS.append({"name": str(name), "ok": bool(ok), "details": str(details)})

PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", PROJECT_ROOT_DEFAULT))
GNINA_BIN = RUNTIME_BIN_DIR / "gnina" if "RUNTIME_BIN_DIR" in globals() else Path("/content/oncotarget_runtime/tools/bin/gnina")
final_manifest_path = PROJECT_ROOT / "redocking_calibration" / "manifests" / "final_docking_jobs_manifest.json"
STEP8_DIR = PROJECT_ROOT / "docking_runs"
STEP8_RUNS_DIR = STEP8_DIR / "runs"
STEP8_REPORTS_DIR = STEP8_DIR / "reports"
STEP8_MANIFESTS_DIR = STEP8_DIR / "manifests"
for p in [STEP8_RUNS_DIR, STEP8_REPORTS_DIR, STEP8_MANIFESTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

with open(final_manifest_path, "r", encoding="utf-8") as f:
    docking_manifest = json.load(f)
all_jobs = docking_manifest.get("docking_jobs", [])

def load_first_sdf_mol(path):
    suppl = Chem.SDMolSupplier(str(path), removeHs=False)
    for mol in suppl:
        if mol is not None:
            return mol
    raise RuntimeError(f"Could not parse any molecule from SDF: {path}")

def extract_top_pose_props(out_sdf):
    mol = load_first_sdf_mol(out_sdf)
    props = {}
    for name in mol.GetPropNames():
        try:
            props[name] = mol.GetProp(name)
        except Exception:
            pass
    return props

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

def run_gnina_docking(job, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    out_sdf = out_dir / "docked.sdf"
    log_txt = out_dir / "gnina.log"
    grid = job["grid"]
    cmd = [
        str(GNINA_BIN), "-r", str(job["receptor_pdb"]), "-l", str(job["ligand_sdf"]),
        "--center_x", str(grid["center_x"]), "--center_y", str(grid["center_y"]), "--center_z", str(grid["center_z"]),
        "--size_x", str(grid["size_x"]), "--size_y", str(grid["size_y"]), "--size_z", str(grid["size_z"]),
        "--cnn_scoring", str(CNN_SCORING_DOCK), "--pose_sort_order", str(POSE_SORT_ORDER_DOCK),
        "--exhaustiveness", str(EXHAUSTIVENESS_DOCK), "--num_modes", str(NUM_MODES_DOCK),
        "--min_rmsd_filter", str(MIN_RMSD_FILTER_DOCK), "--cpu", str(CPU_DOCK), "--seed", str(SEED_DOCK),
        "--out", str(out_sdf), "--log", str(log_txt)
    ]
    if USE_NO_GPU_DOCK:
        cmd.append("--no_gpu")
    res = run_cmd(cmd, check=False)
    if res.returncode != 0:
        raise RuntimeError(f"GNINA docking failed.\nSTDOUT:\n{res.stdout}\nSTDERR:\n{res.stderr}")
    if not out_sdf.exists():
        raise FileNotFoundError(f"GNINA finished but output SDF not found: {out_sdf}")
    return {"out_sdf": str(out_sdf), "log_txt": str(log_txt), "top_pose_props": extract_top_pose_props(out_sdf)}

fcgr_jobs = []
for idx, job in enumerate(all_jobs):
    if job.get("target_input") == TARGET_NAME:
        j = dict(job); j["_global_index"] = idx; fcgr_jobs.append(j)

missing_jobs, completed_jobs = [], []
for job in fcgr_jobs:
    global_idx = job["_global_index"]
    target = job.get("target_input")
    ligand = job.get("ligand_input_name")
    safe_target = sanitize_name(target)
    safe_ligand = sanitize_name(job.get("ligand_query_name") or ligand)
    run_dir = STEP8_RUNS_DIR / f"{global_idx:04d}_{safe_target}__{safe_ligand}"
    report_json = STEP8_REPORTS_DIR / f"{global_idx:04d}_{safe_target}__{safe_ligand}.json"
    out_sdf = run_dir / "docked.sdf"
    if report_json.exists() and out_sdf.exists():
        completed_jobs.append(job)
    else:
        missing_jobs.append(job)

jobs_to_run = missing_jobs[:int(MAX_JOBS_TO_RUN)] if ONLY_IF_MISSING else fcgr_jobs[:int(MAX_JOBS_TO_RUN)]
add_status("Loaded final docking jobs manifest", True, str(final_manifest_path))
add_status("FCGR3A jobs in manifest", True, str(len(fcgr_jobs)))
add_status("FCGR3A completed jobs", True, str(len(completed_jobs)))
add_status("FCGR3A missing jobs detected", True, str(len(missing_jobs)))
add_status("FCGR3A jobs selected for patch", True, str(len(jobs_to_run)))

patch_reports = []
for job in jobs_to_run:
    global_idx = job["_global_index"]
    target = job.get("target_input"); ligand = job.get("ligand_input_name")
    route = job.get("route_type", "unknown")
    safe_target = sanitize_name(target)
    safe_ligand = sanitize_name(job.get("ligand_query_name") or ligand)
    run_dir = STEP8_RUNS_DIR / f"{global_idx:04d}_{safe_target}__{safe_ligand}"
    report_json = STEP8_REPORTS_DIR / f"{global_idx:04d}_{safe_target}__{safe_ligand}.json"
    out_sdf = run_dir / "docked.sdf"
    report = {
        "timestamp": datetime.now().isoformat(),
        "global_job_index": global_idx,
        "target_input": target,
        "ligand_input_name": ligand,
        "ligand_query_name": job.get("ligand_query_name"),
        "route_type": route,
        "receptor_pdb": job.get("receptor_pdb"),
        "ligand_sdf": job.get("ligand_sdf"),
        "grid": job.get("grid"),
        "status": "started"
    }
    try:
        if ONLY_IF_MISSING and report_json.exists() and out_sdf.exists():
            with open(report_json, "r", encoding="utf-8") as f:
                old = json.load(f)
            patch_reports.append(old)
            add_status(f"[{global_idx}] {target} / {ligand}", True, "Skipped (already finished)")
            continue
        result = run_gnina_docking(job, run_dir)
        props = result["top_pose_props"]
        report.update({
            "status": "ok", "out_sdf": result["out_sdf"], "log_txt": result["log_txt"], "top_pose_props": props,
            "CNNscore": safe_float(props.get("CNNscore")), "CNNaffinity": safe_float(props.get("CNNaffinity")),
            "minimizedAffinity": safe_float(props.get("minimizedAffinity")),
        })
        with open(report_json, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2, ensure_ascii=False)
        patch_reports.append(report)
        add_status(f"[{global_idx}] {target} / {ligand}", True, f"CNNscore={report.get('CNNscore')} | CNNaffinity={report.get('CNNaffinity')} | Energy={report.get('minimizedAffinity')}")
    except Exception as e:
        report.update({"status": "error", "error": f"{type(e).__name__}: {e}"})
        with open(report_json, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2, ensure_ascii=False)
        patch_reports.append(report)
        add_status(f"[{global_idx}] {target} / {ligand}", False, f"{type(e).__name__}: {e}")

patch_manifest_path = STEP8_MANIFESTS_DIR / "fcgr3a_auto_patch_run.json"
with open(patch_manifest_path, "w", encoding="utf-8") as f:
    json.dump({
        "timestamp": datetime.now().isoformat(),
        "target_name": TARGET_NAME,
        "missing_jobs_detected": [j["_global_index"] for j in missing_jobs],
        "patched_jobs": patch_reports,
        "status": STATUS
    }, f, indent=2, ensure_ascii=False)
add_status("FCGR3A auto patch manifest saved", True, str(patch_manifest_path))

print("\nReady.")
print("Patch manifest:", patch_manifest_path)
print("\nDetected missing FCGR3A jobs:")
for j in missing_jobs:
    print(f"- [{j['_global_index']}] {j.get('ligand_input_name')} ({j.get('ligand_query_name')})")
print("\nPatch results:")
for row in patch_reports:
    print(f"- [{row.get('global_job_index')}] {row.get('target_input')} / {row.get('ligand_input_name')}: status={row.get('status')} | CNNscore={row.get('CNNscore')} | CNNaffinity={row.get('CNNaffinity')} | Energy={row.get('minimizedAffinity')}")
show_status_panel(STATUS, ok_text="FCGR3A PATCH RUN OK", bad_text="FCGR3A PATCH RUN HAS ERRORS")

## Final aggregation report

In [ ]:
# @title 10. Final aggregation report for all docking results (self-contained)

PROJECT_ROOT_DEFAULT = "/content/drive/MyDrive/oncotarget_pipeline"  # @param {type:"string"}

STATUS = []
def add_status(name, ok, details=""):
    STATUS.append({"name": str(name), "ok": bool(ok), "details": str(details)})

PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", PROJECT_ROOT_DEFAULT))
STEP8_DIR = PROJECT_ROOT / "docking_runs"
STEP8_REPORTS_DIR = STEP8_DIR / "reports"
FINAL_DIR = PROJECT_ROOT / "final_docking_report"
FINAL_TABLES_DIR = FINAL_DIR / "tables"
FINAL_REPORTS_DIR = FINAL_DIR / "reports"
for p in [FINAL_TABLES_DIR, FINAL_REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

all_rows = []
for json_file in sorted(STEP8_REPORTS_DIR.glob("*.json")):
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            obj = json.load(f)
        if "global_job_index" not in obj:
            continue
        all_rows.append(obj)
    except Exception:
        pass

add_status("Docking report JSON files loaded", True, str(len(all_rows)))

by_idx = {}
for row in all_rows:
    idx = row.get("global_job_index")
    if idx is None:
        continue
    if idx not in by_idx:
        by_idx[idx] = row
    else:
        old = by_idx[idx]
        if old.get("status") != "ok" and row.get("status") == "ok":
            by_idx[idx] = row

rows = [by_idx[k] for k in sorted(by_idx.keys())]
ok_rows = [r for r in rows if r.get("status") == "ok"]
bad_rows = [r for r in rows if r.get("status") != "ok"]

add_status("Successful docking jobs", True, str(len(ok_rows)))
add_status("Failed / missing docking jobs", len(bad_rows) == 0, str(len(bad_rows)))

def to_float(x):
    try:
        return float(x)
    except Exception:
        return None

def score_key(row):
    cs = to_float(row.get("CNNscore"))
    ca = to_float(row.get("CNNaffinity"))
    en = to_float(row.get("minimizedAffinity"))
    if cs is None:
        cs = -999.0
    if ca is None:
        ca = -999.0
    if en is None:
        en = 999.0
    return (cs, ca, -en)

ranked_all = sorted(ok_rows, key=score_key, reverse=True)

best_per_target = {}
for row in ranked_all:
    target = row.get("target_input")
    if target not in best_per_target:
        best_per_target[target] = row
best_target_rows = list(best_per_target.values())

top3_per_target = {}
for row in ranked_all:
    target = row.get("target_input")
    top3_per_target.setdefault(target, [])
    if len(top3_per_target[target]) < 3:
        top3_per_target[target].append(row)

md_candidates = []
for target, lst in top3_per_target.items():
    for rank_i, row in enumerate(lst, start=1):
        row2 = dict(row)
        row2["target_rank"] = rank_i
        md_candidates.append(row2)

def save_tsv(rows, path, headers):
    with open(path, "w", encoding="utf-8") as f:
        f.write("\t".join(headers) + "\n")
        for row in rows:
            f.write("\t".join([str(row.get(h, "")) for h in headers]) + "\n")

common_headers = [
    "global_job_index", "target_input", "ligand_input_name", "ligand_query_name",
    "route_type", "status", "CNNscore", "CNNaffinity", "minimizedAffinity",
    "out_sdf", "log_txt"
]

all_results_tsv = FINAL_TABLES_DIR / "all_docking_results.tsv"
ranked_results_tsv = FINAL_TABLES_DIR / "ranked_all_successful.tsv"
best_targets_tsv = FINAL_TABLES_DIR / "best_per_target.tsv"
failed_tsv = FINAL_TABLES_DIR / "failed_or_missing.tsv"
md_candidates_tsv = FINAL_TABLES_DIR / "md_candidates_top3_per_target.tsv"

save_tsv(rows, all_results_tsv, common_headers)
save_tsv(ranked_all, ranked_results_tsv, common_headers)
save_tsv(best_target_rows, best_targets_tsv, common_headers)
save_tsv(bad_rows, failed_tsv, common_headers)
save_tsv(md_candidates, md_candidates_tsv, ["target_rank"] + common_headers)

add_status("All-results TSV saved", True, str(all_results_tsv))
add_status("Ranked-results TSV saved", True, str(ranked_results_tsv))
add_status("Best-per-target TSV saved", True, str(best_targets_tsv))
add_status("Failed/missing TSV saved", True, str(failed_tsv))
add_status("MD-candidates TSV saved", True, str(md_candidates_tsv))

report_md = FINAL_REPORTS_DIR / "final_docking_report.md"
lines = []
lines.append("# Final docking report")
lines.append("")
lines.append(f"Generated: {datetime.now().isoformat()}")
lines.append("")
lines.append(f"- Total docking reports found: {len(rows)}")
lines.append(f"- Successful jobs: {len(ok_rows)}")
lines.append(f"- Failed or missing jobs: {len(bad_rows)}")
lines.append("")
lines.append("## Best ligand per target")
lines.append("")
for row in best_target_rows:
    lines.append(f"- **{row.get('target_input')}** → {row.get('ligand_input_name')} (CNNscore={row.get('CNNscore')}, CNNaffinity={row.get('CNNaffinity')}, Energy={row.get('minimizedAffinity')})")
lines.append("")
lines.append("## Global top 15 successful docking results")
lines.append("")
for i, row in enumerate(ranked_all[:15], start=1):
    lines.append(f"{i}. {row.get('target_input')} / {row.get('ligand_input_name')} (CNNscore={row.get('CNNscore')}, CNNaffinity={row.get('CNNaffinity')}, Energy={row.get('minimizedAffinity')})")
lines.append("")
lines.append("## Top 3 candidates per target for MD")
lines.append("")
for target in sorted(top3_per_target.keys()):
    lines.append(f"### {target}")
    for i, row in enumerate(top3_per_target[target], start=1):
        lines.append(f"- {i}) {row.get('ligand_input_name')} (CNNscore={row.get('CNNscore')}, CNNaffinity={row.get('CNNaffinity')}, Energy={row.get('minimizedAffinity')})")
    lines.append("")
if bad_rows:
    lines.append("## Failed or missing jobs")
    lines.append("")
    for row in bad_rows:
        lines.append(f"- [{row.get('global_job_index')}] {row.get('target_input')} / {row.get('ligand_input_name')} (status={row.get('status')})")
    lines.append("")
report_md.write_text("\n".join(lines), encoding="utf-8")
add_status("Markdown final report saved", True, str(report_md))

summary_json = FINAL_REPORTS_DIR / "final_docking_summary.json"
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump({
        "timestamp": datetime.now().isoformat(),
        "project_root": str(PROJECT_ROOT),
        "n_total_reports": len(rows),
        "n_successful": len(ok_rows),
        "n_failed_or_missing": len(bad_rows),
        "best_per_target": best_target_rows,
        "global_top15": ranked_all[:15],
        "md_candidates_top3_per_target": md_candidates,
        "failed_or_missing": bad_rows,
        "status": STATUS
    }, f, indent=2, ensure_ascii=False)
add_status("JSON final summary saved", True, str(summary_json))

print("\nReady.")
print("All results TSV:", all_results_tsv)
print("Ranked results TSV:", ranked_results_tsv)
print("Best per target TSV:", best_targets_tsv)
print("MD candidates TSV:", md_candidates_tsv)
print("Markdown report:", report_md)
print("JSON summary:", summary_json)

print("\nBest ligand per target:")
for row in best_target_rows:
    print(f"- {row.get('target_input')}: {row.get('ligand_input_name')} | CNNscore={row.get('CNNscore')} | CNNaffinity={row.get('CNNaffinity')} | Energy={row.get('minimizedAffinity')}")

show_status_panel(STATUS, ok_text="FINAL DOCKING REPORT OK", bad_text="FINAL DOCKING REPORT HAS GAPS")

Loaded targets from repaired manifest: 7


NameError: name 'PDBParser' is not defined